# Datathon facility data cleaning

Run the cells from top to bottom using Python compute. The notebook reproduces the reviewed facility and India Post cleaning pipeline from [GitHub](https://github.com/ningyues/cmu-datathon-virtue-healthcare/blob/739dc09d4d763e2f748f968c64bea593266d6aad/pipeline/run_clean.py). It includes the exact address, website, and display-name decision files from that revision.

| Stage | Work | Key outputs |
|---|---|---|
| Bronze | Validate paths and parse the original facility CSV with its notebook evidence | Raw records with source row numbers and evidence keys |
| Silver | Build the postal reference, deduplicate records, standardize fields, validate geography, and apply reviewed corrections | Clean facility and post office CSVs, PIN lookup, claims and review logs |
| Gold | Check row counts, reproduce data-quality metrics, and save a run folder | Data-quality comparison, summaries, and search-ready display names |

**Inputs.** Provide the original **Datathon.zip** workspace export, including the facility CSV and both audit notebooks, and **all_india_pincode_directory_2025(1).csv**. Use files in a Unity Catalog volume you can read and a writable output directory. The original export preserves source order and text for exact replay.

**Scope.** This run covers facility and postal cleaning plus the stored service and website corrections. NFHS-5 remains a separate district-level source. The application uses a separate offline website reachability snapshot.


## Setup · File paths and helpers

Run the first parameter cell to create the widgets. Enter absolute paths, for example `/Volumes/my_catalog/my_schema/my_volume/Datathon.zip`. Set the three values, then run all cells. Each successful run writes to a new directory below the output path.


In [ ]:
import os
from pathlib import Path

parameter_names = ("datathon_zip", "postal_csv", "output_directory")
if "dbutils" in globals():
    for parameter in parameter_names:
        try:
            dbutils.widgets.get(parameter)
        except Exception:
            dbutils.widgets.text(parameter, "", parameter.replace("_", " ").title())
    parameters = {name: dbutils.widgets.get(name).strip() for name in parameter_names}
else:
    parameters = {name: os.environ.get(name.upper(), "").strip() for name in parameter_names}

missing_params = [name for name, value in parameters.items() if not value]
print("Set these paths before running Bronze:", ", ".join(missing_params)) if missing_params else print("Input and output paths configured.")


The shared helpers preserve the source CSV's multiline fields, validate UUIDs and coordinates, normalize known state aliases, and read saved evidence as literal data.


In [ ]:
from collections import Counter, defaultdict
from datetime import date, datetime, timezone
from hashlib import sha256
from io import StringIO
from shutil import copyfile
from tempfile import TemporaryDirectory
from zipfile import ZipFile
import json

import ast
import csv
from io import TextIOWrapper
from math import asin, cos, isfinite, radians, sin, sqrt
import re

FIELDS = ("description", "specialties", "procedure", "equipment", "capability", "capacity")
NIL = {"", "null", "n/a", "na", "unknown"}
UUID_RE = re.compile(r"[0-9a-f]{8}(?:-[0-9a-f]{4}){3}-[0-9a-f]{12}", re.I)
ALIASES = {"nctofdelhi":"delhi", "nctdelhi":"delhi", "nationalcapitalterritoryofdelhi":"delhi",
           "chattisgarh":"chhattisgarh", "telengana":"telangana", "orissa":"odisha",
           "pondicherry":"puducherry", "uttaranchal":"uttarakhand"}


def clean(s):
    return "" if s.strip().lower() in NIL else s.strip()


def compact(s):
    return re.sub(r"[\s.\-]+", "", clean(s).lower())


def state(s):
    key = compact(s.split(",")[-1])
    return ALIASES.get(key, key)


def point(a, b):
    try:
        lat, lon = float(a), float(b)
    except (ValueError, TypeError):
        return None
    if isfinite(lat) and isfinite(lon) and 6 <= lat <= 38 and 68 <= lon <= 98:
        return lat, lon
    return None


def km(a, b):
    x, y, u, v = map(radians, (*a, *b))
    h = sin((x-u)/2)**2 + cos(x)*cos(u)*sin((y-v)/2)**2
    return 12742.0176 * asin(sqrt(min(1, max(0, h))))


def export_rows(z, name):
    with z.open(name) as f, TextIOWrapper(f, encoding="utf-8-sig", newline="") as text:
        reader = csv.reader(text, strict=True)
        header = next(reader)
        if len(set(header)) != len(header) or not set(("unique_id","name",*FIELDS)) <= set(header):
            raise ValueError("The CSV is missing required fields or has duplicate column names")
        last_line = reader.line_num
        rows = []
        for values in reader:
            if len(values) != len(header):
                raise ValueError(f"Unexpected CSV field count at physical line {reader.line_num}")
            rows.append((dict(zip(header, values)), last_line + 1, reader.line_num))
            last_line = reader.line_num
    return header, rows


def evidence(z):
    """Read only data literals from the exported notebooks; never execute their code."""
    src = z.read("Datathon/Datathon/facility_evidence_audit.py").decode("utf-8-sig")
    seeds = {}
    for stmt in ast.parse(src).body:
        if isinstance(stmt, ast.Assign) and isinstance(stmt.targets[0], ast.Name):
            name = stmt.targets[0].id
            if name in {"postal_rows", "field_rows", "reference_date"}:
                seeds[name] = ast.literal_eval(stmt.value)
    postal = {tuple(e[:4]): e[4:] for e in seeds["postal_rows"]}
    guards = {k[0]: k for k in postal}
    facts = {}
    for uid, col, claim, verdict, url, note in seeds["field_rows"]:
        facts[(guards[uid], col, claim)] = (verdict, url, note, seeds["reference_date"])
    # The older SQL notebook also records a named Level III NICU finding.
    # Bind it to the exact branch and SQL evidence tuple; the other SQL finding overlaps the Python notebook.
    sql = z.read("Datathon/Datathon/Datathon-Data Clearning.sql").decode("utf-8-sig")
    branch = ("fac0cc70-81a9-462d-bced-5ca40257b027","Sanjivani Multi Speciality Hospital",
              "Chengannur","Kollakadavu")
    pattern = (r"'"+branch[0]+r"',\s*'capability',\s*'NICU Level III',\s*'official_page_supports',"
               r"\s*'([^']+)',\s*'([^']+)',\s*DATE\s*'([0-9-]+)'")
    match = re.search(pattern, sql)
    if match:
        facts[(branch,"capability","NICU Level III")] = ("official_page_supports", *match.groups())
    return postal, facts


def table(path, fields, rows):
    with path.open("w",encoding="utf-8-sig",newline="") as f:
        w = csv.writer(f)
        w.writerow(fields)
        w.writerows(rows)

PIN = re.compile(r"[1-9][0-9]{5}")
AS_OF = "2026-11-21"
ARRAY_FIELDS = ("specialties", "procedure", "capability", "equipment",
                "source_urls", "websites", "phone_numbers", "source_types",
                "source_ids", "affiliationTypeIds", "countries")
CLAIM_ARRAYS = {"specialties", "procedure", "capability", "equipment"}
DISTRICT_ALIASES = {
    "24 paraganas south": "south 24 parganas", "24 parganas south": "south 24 parganas",
    "24 paraganas north": "north 24 parganas", "24 parganas north": "north 24 parganas",
    "kaimur (bhabua)": "kaimur",
}
EXCLUDED_CLAIMS = {"wrong_facility", "claim_text_names_other_facility"}
SUPPORTED_CLAIMS = {"accreditation_supports_service", "official_page_supports"}


def hash_file(path):
    digest = sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


## Bronze · Load the original snapshots

Read the original facilities and audited evidence. The postal CSV is streamed during Silver to build the reference. Save input fingerprints for comparison with the expected release.


In [ ]:
if missing_params:
    raise ValueError("Set the path widgets, then rerun Setup and Bronze: " + ", ".join(missing_params))

raw_zip = Path(parameters["datathon_zip"]).expanduser()
raw_postal_csv = Path(parameters["postal_csv"]).expanduser()
output_root = Path(parameters["output_directory"]).expanduser()
for name, path in (("datathon_zip", raw_zip), ("postal_csv", raw_postal_csv)):
    if not path.is_absolute() or not path.is_file():
        raise FileNotFoundError(f"{name} must be a readable absolute file path: {path}")
if not output_root.is_absolute():
    raise ValueError("output_directory must be an absolute directory path")

csv.field_size_limit(16_000_000)
with ZipFile(raw_zip) as archive:
    facility_columns, bronze_facilities = export_rows(archive, "Datathon/facilities.csv")
    prior_pins, prior_claims = evidence(archive)

input_fingerprints = {
    "datathon_zip": hash_file(raw_zip),
    "postal_csv": hash_file(raw_postal_csv),
}
print(f"Loaded {len(bronze_facilities):,} source facility records.")
print(f"Loaded {len(prior_pins)} postal findings and {len(prior_claims)} reviewed claims.")
print(json.dumps(input_fingerprints, indent=2))


## Silver · Reviewed corrections

The embedded decision tables contain the reviewed values and evidence links. Facility and name corrections require both the matching UUID and the expected source value. Postal corrections also check the recorded office name, state, and district.


In [ ]:
POSTAL_SOURCE_DECISIONS_CSV = (
    'pincode,officename,expected_state,expected_district,state,district,source_url,note\n'
    '799001,Jaganath Bari Road S.O,TELANGANA,Nirmal,TRIPURA,West Tripura,https://westtripura.nic.in/public-utility/agartala-postal-ho/,"District administration confirms PIN 799001 in Agartala, West Tripura; office is in Agartala Division."\n'
    '799001,Agartala H.O,TELANGANA,Nirmal,TRIPURA,West Tripura,https://westtripura.nic.in/public-utility/agartala-postal-ho/,"District administration confirms PIN 799001 in Agartala, West Tripura; office is in Agartala Division."\n'
    '799001,Agartala Bazar S.O,TELANGANA,Nirmal,TRIPURA,West Tripura,https://westtripura.nic.in/public-utility/agartala-postal-ho/,"District administration confirms PIN 799001 in Agartala, West Tripura; office is in Agartala Division."\n'
    '799001,Agartala Court S.O,TELANGANA,Nirmal,TRIPURA,West Tripura,https://westtripura.nic.in/public-utility/agartala-postal-ho/,"District administration confirms PIN 799001 in Agartala, West Tripura; office is in Agartala Division."\n'
)


In [ ]:
FACILITY_SOURCE_DECISIONS_CSV = (
    'id,field,expected,value,status,source_url,corroborating_url,note\n'
    '9a7c8cd7-5c16-427e-8e14-2a5c4efa1231,address_zipOrPostcode,641022,641043,source_backfill,https://www.gangahospital.com/help-desk,https://www.tnprivatejobs.tn.gov.in/candidate/Home/employer_profile_single/2240207155555518,"Same street number 313, Mettupalayam Road, Coimbatore in the official and Tamil Nadu portal records"\n'
    '7dbdbeca-0f56-467b-a822-77c30af1fff1,address_zipOrPostcode,451220,452001,source_backfill,https://suyashhospital.com/contact-us/,,Indore A.B. Road branch confirmed by the facility contact page and website already attached to the input\n'
    'fac0cc70-81a9-462d-bced-5ca40257b027,address_stateOrRegion,Alappuzha,Kerala,source_backfill,https://smsh.in/book-appointment/,https://portal.nabh.co/Documents/AccreditedList/Entry-Level%20Hospitals/PEH-2021-1651.pdf,Both sources match Kollakadavu/Chengannur PIN 690509. Alappuzha is the district; Kerala is the state\n'
    '77a4e680-bf90-4cd0-b4b5-32bcdd7cf61c,capacity,275,,scope_conflict,https://portal.nabh.co/Documents/AccreditedList/Hospitals/H-2015~3.PDF,,Opposite Mahamarg Bus Stand certificate gives 75 sanctioned beds. Scope of original 275 unclear\n'
    '77a4e680-bf90-4cd0-b4b5-32bcdd7cf61c,sanctioned_beds,,75,source_backfill,https://portal.nabh.co/Documents/AccreditedList/Hospitals/H-2015~3.PDF,,NABH H-2015-0315 certification 2025-08-03 through 2029-08-02; fill only the sanctioned-beds field\n'
    'b8a5401f-42f1-422a-8cd9-686a15b4cb76,address_zipOrPostcode,110002,,mixed_branch_pending,https://www.aravindeyecarehospitals.com/contact/,https://igh.delhi.gov.in/igh/services-and-facilities,Record mixes Hyderabad and New Delhi content. Hold branch PIN correction pending disambiguation\n'
    '3ac5ae50-9ce2-4de3-a13c-d95a4ef218ee,address_zipOrPostcode,400078,,source_conflict,https://www.wockhardthospitals.com/contact-us/,https://wockhardthospitals.com/international-patients/contact-us/,Nagpur hospital contact pages give 440033 and 440010 respectively; remove 400078 and keep both candidates unresolved\n'
    '3ac5ae50-9ce2-4de3-a13c-d95a4ef218ee,officialWebsite,fortishealthcare.com,wockhardthospitals.com,source_backfill,https://www.wockhardthospitals.com/contact-us/,https://wockhardthospitals.com/international-patients/contact-us/,Official pages identify this Nagpur Wockhardt branch; both agree on operator website\n'
    '51179ca4-f342-4014-92d7-c2d2ef1adbb1,address_stateOrRegion,Andhra Pradesh,Telangana,source_backfill,https://dme.telangana.gov.in/open_record.php?ID=196,,Telangana 政府名单匹配 Netaji Chowk、TNGO’s Road；保留原 PIN 作邮政一致值。坐标在 East Godavari。\n'
    '7e08f70c-6fc4-49f8-a1e9-fdddda2e0469,address_city,Hyderabad,Visakhapatnam,source_backfill,https://portal.nabh.co/Documents/AccreditedList/Hospitals/H-2021-0883_5th%20Edition.pdf,,NABH 文件以 Plot 4、Chinnagadili 定位 Vizag 院区，PIN 相符。文件仅用于核对地点。原坐标在 Hyderabad。\n'
    '7e08f70c-6fc4-49f8-a1e9-fdddda2e0469,address_stateOrRegion,Telengana,Andhra Pradesh,source_backfill,https://portal.nabh.co/Documents/AccreditedList/Hospitals/H-2021-0883_5th%20Edition.pdf,,NABH 文件以 Plot 4、Chinnagadili 定位 Vizag 院区，PIN 相符。文件仅用于核对地点。原坐标在 Hyderabad。\n'
    'ecd07024-da17-4193-ad64-f67139165116,address_city,Hyderabad,Visakhapatnam,source_backfill,https://www.carehospitals.com/contact-us,,官网 Ramnagar 院区的 AS Raja Complex、10-50-11/5 和 Waltair Main Road 与原门牌相符，原 PIN 保留。原坐标在 Hyderabad。\n'
    'ecd07024-da17-4193-ad64-f67139165116,address_stateOrRegion,Telangana,Andhra Pradesh,source_backfill,https://www.carehospitals.com/contact-us,,官网 Ramnagar 院区的 AS Raja Complex、10-50-11/5 和 Waltair Main Road 与原门牌相符，原 PIN 保留。原坐标在 Hyderabad。\n'
    'aee90e94-5d26-4247-a0c5-7db96b6839ad,address_stateOrRegion,Delhi,Haryana,source_backfill,https://mayomhospital.com/contact,https://www.icssr.org/sites/default/files/2026-03/List-of-Hospitals-Empanelled-as-on-11-03-2026.pdf,官网和 ICSSR 名录都写 D Block、South City I、Haryana 122001。地图的邻近邮编不可覆盖院方地址。\n'
    'e667b954-ba77-4af8-8388-6aebb067401c,address_city,Nagar,Kota,source_backfill,https://esic.gov.in/attachments/tenderfile/3d658dc7cfe4762e955bdfd2104f16fd.pdf,,ESIC 官方文件明确是 Kota 的 Jhalawar Road，坐标落在 Kota。原城市 Nagar 与 PIN 321205 需替换。\n'
    'e667b954-ba77-4af8-8388-6aebb067401c,address_stateOrRegion,Haryana,Rajasthan,source_backfill,https://esic.gov.in/attachments/tenderfile/3d658dc7cfe4762e955bdfd2104f16fd.pdf,,ESIC 官方文件明确是 Kota 的 Jhalawar Road，坐标落在 Kota。原城市 Nagar 与 PIN 321205 需替换。\n'
    'e667b954-ba77-4af8-8388-6aebb067401c,address_zipOrPostcode,321205,324005,source_backfill,https://esic.gov.in/attachments/tenderfile/3d658dc7cfe4762e955bdfd2104f16fd.pdf,,ESIC 官方文件明确是 Kota 的 Jhalawar Road，坐标落在 Kota。原城市 Nagar 与 PIN 321205 需替换。\n'
    '8aca0f97-c29e-47d2-9584-b2405452aa33,address_stateOrRegion,Andhra Pradesh,Telangana,source_backfill,https://www.moil.nic.in/userfiles/PAN%20INDIA%20NETOWRK_HOSPITALS_13-07-2025%282%29.pdf,,网络医院名单匹配 Jeedimetla 的 APIIC Colony、Opposite Sub Station。保留该门牌与 PIN，更新州。\n'
    '0cf3edb4-cd74-4395-839d-4f1a99f783a7,address_stateOrRegion,Andhra Pradesh,Telangana,source_backfill,https://health.py.gov.in/sites/default/files/empanelledhospitals.pdf,https://hrms.pnb.bank.in/pdf/HITPA%20HOSPITAL%20LIST1.pdf,公开名单匹配 3-4-852 Barkatpura 和 PIN。两份名单用于地址核对。\n'
    '2e4137a3-6a58-434a-a2eb-67dfd31bdff0,address_city,Ongole,Hyderabad,source_backfill,https://www.ravihospitals.com/slider/slider5/,,官网明确 MIG-549、Road 1、KPHB Colony。应保留这家 Hyderabad 院区，改城市、州和 PIN。\n'
    '2e4137a3-6a58-434a-a2eb-67dfd31bdff0,address_stateOrRegion,Andhra Pradesh,Telangana,source_backfill,https://www.ravihospitals.com/slider/slider5/,,官网明确 MIG-549、Road 1、KPHB Colony。应保留这家 Hyderabad 院区，改城市、州和 PIN。\n'
    '2e4137a3-6a58-434a-a2eb-67dfd31bdff0,address_zipOrPostcode,500085,500072,source_backfill,https://www.ravihospitals.com/slider/slider5/,,官网明确 MIG-549、Road 1、KPHB Colony。应保留这家 Hyderabad 院区，改城市、州和 PIN。\n'
    'bc3bb6d6-419c-4db9-94d9-63f61e1d72a1,address_stateOrRegion,Chandigarh,Punjab,source_backfill,https://www.dhsr.hp.gov.in/Files/Final%20list%20EMP%2006.01.2026.pdf,,2026 年政府名单匹配 Phase 4 与 Near Telephone Exchange，列 160059。采用同门牌对应邮编。\n'
    'bc3bb6d6-419c-4db9-94d9-63f61e1d72a1,address_zipOrPostcode,160055,160059,source_backfill,https://www.dhsr.hp.gov.in/Files/Final%20list%20EMP%2006.01.2026.pdf,,2026 年政府名单匹配 Phase 4 与 Near Telephone Exchange，列 160059。采用同门牌对应邮编。\n'
    '5cb454ef-4ff7-4948-b832-511ae126e526,address_stateOrRegion,Andhra Pradesh,Telangana,source_backfill,https://tanvirhospital.com/home/,,官网的 8-3-833、Plot 100 匹配原门牌，明确写 Telangana 500073。\n'
    'deb837c2-e6b5-490e-90a2-e5adf36ee69e,address_stateOrRegion,Andhra Pradesh,Telangana,source_backfill,https://nikhilhospitals.com/dilsukhnagar/,,官网区分 Dilsukhnagar 与 Ameerpet；原地址 Gaddi Annaram 匹配 Dilsukhnagar 500036。\n'
    'efb047fe-1b25-48a4-b51a-b5a2389625c2,address_stateOrRegion,Andhra Pradesh,Telangana,source_backfill,https://www.tsso.in/pdf/New%20updated%20TG%20Recognized%20Pvt.%20Hospitals%20List-2023-2025.pdf,https://www.iffcotokio.co.in/content/dam/iffcotokio/iffco-pdf/sites/default/files/Web%20LIST%20WEF_20-Dec-2019%20PDF.pdf,识别名录匹配 Plot 2 & 3、Peerzadiguda。保险方名单也给相同街区及 PIN；不以过期条款判断当前接诊。\n'
    '9201d9c9-4ebc-4a21-bd47-9869d95e016f,address_stateOrRegion,Chhattisgarh,Telangana,source_backfill,https://www.cloudninecare.com/centre/banjara-hills,,官方 Banjara Hills 院区页匹配 Road No. 2 和原 PIN；州改 Telangana。\n'
    '9c33d229-7161-44d0-b632-4c5cd2b9a1d0,address_stateOrRegion,Andhra Pradesh,Telangana,source_backfill,https://uiic.co.in/sites/default/files/uploads/ppn_network_hospital/hyderabad_ppn_31-7-2023_0.pdf,https://hrms.pnb.bank.in/pdf/HITPA%20HOSPITAL%20LIST1.pdf,两份保险资料区分 Barkatpura 分院且列 500027 Telangana；坐标落在另一行政区。\n'
    'cf6d8989-4a44-419d-b497-a8c91b35b098,address_zipOrPostcode,520010,590001,source_backfill,https://belagavi.nic.in/en/public-utility-category/hospitals/,https://kla.kar.nic.in/council/Replies/unstarred/10032025/usq482%2810032025%29.pdf,区政府医院列表确认 Ashoknagar、Belagavi 590001，立法机关资料可对到门牌。原 520010 移除。\n'
    'faf71fe2-fe08-4642-9e6e-09b1f98c7d36,address_city,Ghaziabad,Faridabad,source_backfill,https://www.seedsofinnocens.com/contact/centre-locator/,,官网定位器把 A-6/A Neelam Bata Road、ICICI 上方列在 Faridabad。保留地址和属地相符的原 PIN，修正 Ghaziabad 的城市／州字段，设施名含城市也需标注。\n'
    'faf71fe2-fe08-4642-9e6e-09b1f98c7d36,address_stateOrRegion,Uttar Pradesh,Haryana,source_backfill,https://www.seedsofinnocens.com/contact/centre-locator/,,官网定位器把 A-6/A Neelam Bata Road、ICICI 上方列在 Faridabad。保留地址和属地相符的原 PIN，修正 Ghaziabad 的城市／州字段，设施名含城市也需标注。\n'
    '076633ac-580a-42b0-8bb6-94828bd6ccfe,address_stateOrRegion,Punjab,Chandigarh,source_backfill,https://www.indiraivf.com/fertility-doctors/dr-itisha-gupta,https://chandigarh.gov.in/redefining-chandigarh,政府登记和院方同为 SCO 156–159、Sector 9C。Chandigarh 为独立联邦属地，院方页面的 Punjab 属栏目标签或误填。\n'
    '8c0ec23b-2d96-4fd8-93dd-636129a77b29,address_stateOrRegion,Delhi,Uttar Pradesh,source_backfill,https://www.bis.gov.in/wp-content/uploads/2025/07/circular-194-consolidated-list-16-July-2025.pdf,https://mowr.nic.in/core/Circulars/2022/GA_26-07-2022_33.pdf,政府名单为 B-1A/22 Ground Floor Sector 51、Noida 201301；对应原地址和 geo。Delhi 名录另有一所分点。\n'
    '92d193b6-e76a-47b5-a095-1079af154ef0,address_stateOrRegion,Andhra Pradesh,Telangana,source_backfill,https://www.sankhyahospitals.com/contact,,官网匹配 Sy No.1009、BSNL Office、KPHB，明确 Telangana 500072。\n'
    '9cabadda-a8b5-4faa-887d-9ea1b73e6788,address_stateOrRegion,Andhra Pradesh,Telangana,source_backfill,https://www.icicilombard.com/docs/default-source/apps/elevate-client-app/assets/pdf/activeppnlist_28may24-edited-sj2.pdf,,保险方可对到 15-1-29、Srinivasa Colony、Warangal，PIN 与原始一致。\n'
    'dca537aa-90cc-4564-8eca-a135e27a8fa9,address_stateOrRegion,Andhra Pradesh,Telangana,source_backfill,https://health.py.gov.in/sites/default/files/empanelledhospitals.pdf,,政府公开的医院名单匹配 Himayath Nagar、B-6-756 Street 13。州改 Telangana，PIN 与原州候选一致。\n'
    '84257413-6cc5-4a88-99ef-2dda1148f01c,address_stateOrRegion,Andhra Pradesh,Telangana,source_backfill,https://punjabandsindbank.co.in/system/uploads/document/2150_2023032413224150735.pdf,,公开保险清单匹配 9-4-87/5/2、Tolichowki；当地门牌与 PIN 相符，旧点仍需另核。\n'
    '51179ca4-f342-4014-92d7-c2d2ef1adbb1,latitude,16.956157684326172,,coordinate_location_conflict,https://dme.telangana.gov.in/open_record.php?ID=196,,Telangana 政府名单匹配 Netaji Chowk、TNGO’s Road；保留原 PIN 作邮政一致值。坐标在 East Godavari。\n'
    '51179ca4-f342-4014-92d7-c2d2ef1adbb1,longitude,82.23306274414062,,coordinate_location_conflict,https://dme.telangana.gov.in/open_record.php?ID=196,,Telangana 政府名单匹配 Netaji Chowk、TNGO’s Road；保留原 PIN 作邮政一致值。坐标在 East Godavari。\n'
    '51179ca4-f342-4014-92d7-c2d2ef1adbb1,coordinates,"{""coordinates"":[82.23306274414062,16.956157684326172],""type"":""Point""}",,coordinate_location_conflict,https://dme.telangana.gov.in/open_record.php?ID=196,,Telangana 政府名单匹配 Netaji Chowk、TNGO’s Road；保留原 PIN 作邮政一致值。坐标在 East Godavari。\n'
    '7e08f70c-6fc4-49f8-a1e9-fdddda2e0469,latitude,17.41378402709961,,coordinate_location_conflict,https://portal.nabh.co/Documents/AccreditedList/Hospitals/H-2021-0883_5th%20Edition.pdf,,NABH 文件以 Plot 4、Chinnagadili 定位 Vizag 院区，PIN 相符。文件仅用于核对地点。原坐标在 Hyderabad。\n'
    '7e08f70c-6fc4-49f8-a1e9-fdddda2e0469,longitude,78.42445373535156,,coordinate_location_conflict,https://portal.nabh.co/Documents/AccreditedList/Hospitals/H-2021-0883_5th%20Edition.pdf,,NABH 文件以 Plot 4、Chinnagadili 定位 Vizag 院区，PIN 相符。文件仅用于核对地点。原坐标在 Hyderabad。\n'
    '7e08f70c-6fc4-49f8-a1e9-fdddda2e0469,coordinates,"{""coordinates"":[78.42445373535156,17.41378402709961],""type"":""Point""}",,coordinate_location_conflict,https://portal.nabh.co/Documents/AccreditedList/Hospitals/H-2021-0883_5th%20Edition.pdf,,NABH 文件以 Plot 4、Chinnagadili 定位 Vizag 院区，PIN 相符。文件仅用于核对地点。原坐标在 Hyderabad。\n'
    'ecd07024-da17-4193-ad64-f67139165116,latitude,17.385581970214844,,coordinate_location_conflict,https://www.carehospitals.com/contact-us,,官网 Ramnagar 院区的 AS Raja Complex、10-50-11/5 和 Waltair Main Road 与原门牌相符，原 PIN 保留。原坐标在 Hyderabad。\n'
    'ecd07024-da17-4193-ad64-f67139165116,longitude,78.47396850585938,,coordinate_location_conflict,https://www.carehospitals.com/contact-us,,官网 Ramnagar 院区的 AS Raja Complex、10-50-11/5 和 Waltair Main Road 与原门牌相符，原 PIN 保留。原坐标在 Hyderabad。\n'
    'ecd07024-da17-4193-ad64-f67139165116,coordinates,"{""coordinates"":[78.47396850585938,17.385581970214844],""type"":""Point""}",,coordinate_location_conflict,https://www.carehospitals.com/contact-us,,官网 Ramnagar 院区的 AS Raja Complex、10-50-11/5 和 Waltair Main Road 与原门牌相符，原 PIN 保留。原坐标在 Hyderabad。\n'
    '02b010de-143d-4e1b-9d67-cb11877c4855,address_stateOrRegion,Mumbai,Maharashtra,source_backfill,https://www.karunahospitalmsa.com/contact.html,,"Jeevan Bima Nagar near Bhagwati Hospital, Borivali West, Mumbai 400103 matches the branch. The postal directory assigns the PIN to Maharashtra."\n'
    '082c1e59-7d97-4dec-ba8b-0c3d4ce4bb43,address_stateOrRegion,Kollam,Kerala,source_backfill,https://www.holycrosskottiyam.org/,https://portal.nabh.co/Documents/AccreditedList/Entry-Level%20Hospitals/PEH-2020-1293.pdf,"The hospital and NABH list the Kottiyam campus in Kollam, Kerala 691571. The alternate street fragments were not independently resolved."\n'
    '1c3f0043-bd07-44ea-854c-f9065dbfb0ac,address_stateOrRegion,Pune,Maharashtra,source_backfill,https://www.adityabirlahospital.com/contact-us,https://portal.nabh.co/Documents/AccreditedList/Hospitals/H-2013-0169_5th%20Edition_Socpe%20Correction_SA.pdf,"Both sources place Aditya Birla Hospital Marg, Chinchwad 411033 in Maharashtra."\n'
    '231f141f-cc05-42a9-949a-accf4bab933c,address_zipOrPostcode,430115,431005,source_backfill,https://ciigmagroup.org/google-maps/,https://portal.nabh.co/Documents/AccreditedList/Hospitals/H-2016-0418_5th%20Edition.pdf,"The hospital location page matches plots 6 and 7, Shahnoorwadi Dargah Road and lists 431005."\n'
    '717c6f01-7802-4cb6-838e-3fbd0063b201,address_zipOrPostcode,700098,700106,source_backfill,https://mahi.manipalhospitals.com/kolkata/health-checkups/,https://portal.nabh.co/Documents/AccreditedList/Entry-Level%20Hospitals/PEH-2023-2207_Revised.pdf,"Manipal identifies the Broadway Road and stadium-gate site. The AMRI certificate identifies JC 16 and 17, Sector 3, Salt Lake and the same PIN."\n'
    'e0b3c833-62ed-46b9-aa05-115b9c9fb400,address_zipOrPostcode,75,500075,source_backfill,https://www.bhaskarmedicalcollege.edu.in/pdf-files/Pro-forma-18.pdf,https://www.bhaskarmedicalcollege.edu.in/,"The college\'s disclosure identifies Bhaskar General Hospital and lists its Yenkapally, Moinabad campus as Telangana 500075."\n'
    '32f97788-fe6d-4a3a-8129-f8e9a3e87ef0,address_zipOrPostcode,58,700058,source_backfill,https://www.esic.gov.in/attachments/tenderfile/Annexure-08-ESIC-Network-Connectivity-Details.pdf,https://www.wbpcb.gov.in/files/Tu-02-2025-02-32-31ESI%20Hospital.pdf,"ESIC lists the Kamarhati ESI Hospital campus, 1 Graham Lane, West Bengal 700058. The state regulator corroborates the hospital site."\n'
    '5ab66909-41a2-4699-969c-3cbe8b127a6b,address_stateOrRegion,New Mondha,Maharashtra,source_backfill,https://health.py.gov.in/sites/default/files/empanelledhospitals.pdf,https://kwa.kerala.gov.in/wp-content/uploads/2024/07/PEN-LIST-OF-NETWORK-HOSPITALS_PAN-_INDIA_Ms-ORIENTAL-INSURANCE-COMPANY-LTD.pdf,"Both listings match New Doctor Lane, Bus Stand Road, Parbhani, Maharashtra. The second also lists 431401."\n'
    '60477c08-33c4-4b95-b38f-545ddd23cdfb,address_zipOrPostcode,5600,560052,source_backfill_unlisted_pin,https://portal.nabh.co/Documents/AccreditedList/Hospitals/H-2020-0686_5th%20Edition.pdf,https://www.indiapost.gov.in/MBE/DOP_PDFFiles/Alphabetical_Index_of_Delivery_Offices_on_Speed_Post_Network.pdf,"NABH identifies the Millers Road, Vasanth Nagar branch and PIN. India Post lists that PIN. It is absent from the supplied coordinate snapshot."\n'
    '7c385d20-6a63-417b-a336-154bd22a60e7,address_zipOrPostcode,000000,302015,source_backfill,https://portal.nabh.co/Documents/AccreditedList/Hospitals/H-2025-1624_5th%20Edition.pdf,https://jaipurhospital.co.in/wp-content/uploads/2024/09/Notice-Postal-Ballot.pdf,"Both sources identify the Lal Kothi site near SMS Stadium as Jaipur, Rajasthan 302015."\n'
    '9a1d66f9-196a-4ce2-9baf-f855b8b8fb29,address_stateOrRegion,Thrissur District,Kerala,source_backfill,https://esic.gov.in/attachments/sstfile/LIST_OF_EMPANELLED_HOSPITALS_FOR_SUPER_SPECIALITY_SERVICES_TO_ESI_BENEFICIARIES_IN_KERALA_REGION_1773660180.pdf,,"Kerala-region ESIC directory matches Modern Hospital, post box 22, Kodungallur, Thrissur 680664."\n'
    'b19e36f0-6fb6-44ab-b86c-beba737ba6e8,address_stateOrRegion,Gandhidham,Gujarat,source_backfill,https://ipc.gov.in/images/mvpi/List-of-725-MDMCs.pdf,,"Indian Pharmacopoeia Commission directory matches Ward 2A, Post Office Road, near Gandhi Samadhi, Adipur, Kutch, Gujarat 370205."\n'
    '123421c9-86f6-4a45-8a27-00eab19c32f4,address_stateOrRegion,New Delhi,Delhi,source_backfill,https://www.prlworld.com/contact-us.aspx,https://nabl-india.org/nabl/file_download1.php?filename=202402200749-NABL-600-doc.pdf,"The lab and NABL identify 515, Sector 19, Dwarka, Delhi 110075."\n'
    '322e4037-ac32-4b53-a2d7-2ef6ea10a75b,address_stateOrRegion,Chadayamangalam,Kerala,source_backfill,https://sha.kerala.gov.in/?page_id=563,https://arafahospitals.in/,"Kerala State Health Agency lists Arafa Medical Mission Hospital in Kadakkal, Kollam 691536. The hospital confirms the town."\n'
    '42d62a1a-4b9c-4b6a-92ee-e5f9bf10718e,address_stateOrRegion,Kutch,Gujarat,source_backfill,https://www.hmcmandvi.com/contact-us,https://www.hmcmandvi.com/,"The hospital\'s contact page matches Nagalpur Road near Dada Ni Deri and identifies Mandvi, Kutch, Gujarat 370465."\n'
    '4c536d13-417c-4b80-a3dd-5995db8f9aad,address_zipOrPostcode,641-602,641602,source_backfill,https://sreesaranmedicalcenter.com/contact-us/,https://kwa.kerala.gov.in/wp-content/uploads/2024/07/PEN-LIST-OF-NETWORK-HOSPITALS_PAN-_INDIA_Ms-ORIENTAL-INSURANCE-COMPANY-LTD.pdf,"The facility contact page matches SF 259, Poyampalayam Pirivu, P.N. Road, Tirupur 641602."\n'
    '5cfbd727-caa8-4c86-b411-4187cd95c26b,address_stateOrRegion,Dharwad,Karnataka,source_backfill,https://kwa.kerala.gov.in/wp-content/uploads/2024/07/PEN-LIST-OF-NETWORK-HOSPITALS_PAN-_INDIA_Ms-ORIENTAL-INSURANCE-COMPANY-LTD.pdf,https://kwa.kerala.gov.in/wp-content/uploads/2024/07/Employees-LIST-OF-NETWORKS-HOSPITALS_PAN-INDIA-_Ms-United-India-Insurance-Company.pdf,"Two insurer directories match Vivekanand General Hospital, Deshpande Nagar, Hubli 580029. The postal directory assigns the PIN to Karnataka."\n'
    '681c3935-f3e2-44b1-95ca-a27ebffcabae,address_stateOrRegion,Idukki,Kerala,source_backfill,https://holyfamilyhospitals.com/contact-us/,https://kwa.kerala.gov.in/wp-content/uploads/2026/05/GHIS-2026-27-Employees-Hospital-list-in-Kerala.pdf,"The hospital and published scheme directory identify Muthalakodam, Thodupuzha, Idukki, Kerala 685605."\n'
    '037a873f-cae9-4491-b344-e723b6b9f961,address_stateOrRegion,Dakshin Kannad,Karnataka,source_backfill,https://portal.nabh.co/Documents/AccreditedList/Entry-Level%20Hospitals/PEH-2020-1323.pdf,,"NABH lists the Falnir Road, Mangalore branch in Dakshina Kannada, Karnataka 575002."\n'
    '2ac539eb-eaff-4335-8e13-4f1d379d31b5,address_stateOrRegion,Mohali,Punjab,source_backfill,https://www.indushospital.in/contact/contact-us,https://portal.nabh.co/Documents/AccreditedList/Hospitals/H-2018-0537_5th%20Edition.pdf,"Both sources identify NH 22, Dera Bassi, Mohali, Punjab 140507."\n'
    '48ea2054-d2c8-4824-975a-cd692b6f6f52,address_stateOrRegion,Navi Mumbai,Maharashtra,source_backfill,https://www.icicilombard.com/docs/default-source/apps/elevate-client-app/assets/pdf/activeppnlist_28may24-edited-sj2.pdf,,"The insurer lists plots 108–109, opposite Shiv Sena Shakha, Sector 11, Vashi, Maharashtra 400703. The postal directory agrees."\n'
    '69201515-916a-4304-aeef-e700f8603083,address_zipOrPostcode,62,110062,source_backfill,https://www.batrahospitaldelhi.org/contact-us,https://portal.nabh.co/Documents/AccreditedList/Hospitals/H-2010-0051_5th%20Edition.pdf,The current contact page and later NABH scope agree on 110062 at 1 Tughlakabad Institutional Area. A 2021 facility report listed 110044.\n'
    '6a092efb-cd6d-4378-9075-e8eaba16e3dd,address_stateOrRegion,Srinagar Kashmir,Jammu and Kashmir,source_backfill,https://noorahospital.org/contact-us/,https://portal.nabh.co/Documents/AccreditedList/Entry-Level%20Hospitals/PEH-2023-2405.pdf,"Both sources identify Umerabad, Zainakote, HMT, Srinagar 190012. Expand the union territory name used by the postal directory."\n'
    'a35243f6-e2f2-4b59-a784-95225f0669c4,address_zipOrPostcode,141009,141003,source_backfill,https://spshospitals.com/terms-and-conditions/,https://spshospitals.com/contacts/,"The hospital identifies the main Sherpur Chowk, Grand Trunk Road campus and explicitly lists Punjab 141003 in its terms."\n'
    '0277b0a0-05e0-418d-9763-7b2fe78f636e,address_zipOrPostcode,2110...,211001,source_backfill,https://srijanhospital.com/contact.php,https://www.cbse.gov.in/cbsenew/documents/medical%20policy%202018.pdf,"Both sources match Elgin Road, Civil Lines opposite Prayag Mahila Vidyapeeth and list 211001."\n'
    '1acced00-8ab1-4e2f-8d40-bf6d5218892b,address_stateOrRegion,U.p.,Uttar Pradesh,source_backfill,https://www.zaubacorp.com/company-by-address/SHOP-NO-18--MONNI-MONTA-MARKET-LUCKNOW-LUCKNOW-UTTAR-PRADESH-INDIA-226014,https://www.planetexim.net/indian-company/p-k-pathology-and-xray-lab-llp/llpin/ACP-6011.html,"Company listings match Monni Monta Market near SGPGI, Lucknow 226014. Expand the source state abbreviation U.p. to the postal directory\'s state."\n'
    '2bc5af49-5cbb-4cf9-9009-1b5d861ac02e,address_stateOrRegion,Palghar,Maharashtra,source_backfill,https://cardinalgraciashospital.com/dnb/,https://cardinalgraciashospital.com/contact-us/,"The hospital places its Vasai West, Palghar 401201 campus in Maharashtra."\n'
    '9f415a94-7c72-473a-ab96-ad63a8910b3d,address_stateOrRegion,Malappuram,Kerala,source_backfill,https://moulanahospital.com/contact-us/,,"The hospital distinguishes its Perintalmanna main campus, post box 31, Kerala 679322 from its other site."\n'
    'de46c638-bd08-4aab-b5f8-31c4def8851a,address_stateOrRegion,Jalandhar-east,Punjab,source_backfill,https://portal.nabh.co/Documents/AccreditedList/Hospitals/H-2025-1572_5th%20Edition.pdf,https://innocentheartssuperspeciality.com/contact-us/,"NABH and the facility identify the Nakodar Road site near Wadala Chowk, Jalandhar 144003."\n'
    'e306b567-daee-42b0-87dd-cc251fdff77c,address_stateOrRegion,Anchal,Kerala,source_backfill,https://bharathhospital.co.in/contact-us,https://portal.nabh.co/Documents/AccreditedList/Hospitals/H-2025-1719_6th%20Edition.pdf,"The facility domain, name, Azad Lane and PIN 686001 match the Kottayam facility in Kerala. This replaces conflicting locality and state values."\n'
    'e306b567-daee-42b0-87dd-cc251fdff77c,address_city,Kulathupuzha,Kottayam,source_backfill,https://bharathhospital.co.in/contact-us,https://portal.nabh.co/Documents/AccreditedList/Hospitals/H-2025-1719_6th%20Edition.pdf,"The facility domain, name, Azad Lane and PIN 686001 match the Kottayam facility in Kerala. This replaces conflicting locality and state values."\n'
    '1bdc33d0-a5ca-4d7c-a000-6da9336c9efb,address_stateOrRegion,Chikhali,Maharashtra,source_backfill,https://imperialhospitals.in/,https://kwa.kerala.gov.in/wp-content/uploads/2024/07/PEN-LIST-OF-NETWORK-HOSPITALS_PAN-_INDIA_Ms-ORIENTAL-INSURANCE-COMPANY-LTD.pdf,"Both sources identify the Pingale Road, Chikhali, Pune 411062 campus in Maharashtra."\n'
    '40731bc2-2799-4300-9b90-df3f2d771706,address_stateOrRegion,Kachchh,Gujarat,source_backfill,https://www.kkphospital.org/public/images/academics/ECMO_Critical_Care_N_Fellowship_Program_2025.pdf,https://www.pib.gov.in/PressReleaseIframePage.aspx?PRID=1816545&lang=2&reg=48,"Hospital publication confirms the site near Kutch University on Mundra Road, Bhuj, Gujarat 370001."\n'
    '971cff56-dd03-488f-9efd-8483887361b5,address_stateOrRegion,Bhavnagar,Gujarat,source_backfill,https://kwa.kerala.gov.in/wp-content/uploads/2021/12/Pension-Oriental-Insurance-Co.-Ltd.pdf,,"The insurer\'s published listing matches Gopal Arcade, near Top 3 Circle, Talaja Road, Adhewada, Gujarat 364002."\n'
    'a2b6c40b-0d96-4484-b999-2f35b96d04ec,address_stateOrRegion,Andhra Pradesh,Telangana,source_backfill,https://www.magmainsurance.com/documents/5553002/0/List%20of%20Non-network%20hospital.pdf/df6ed795-a41e-5816-dd91-637504cb72bc,https://dggi.cbic.gov.in/dggi-ui/zonal-units,"Insurer address matches 1-66/4/194, Block B, Kavuri Hills, Madhapur, Telangana 500033. Government corroboration is for the locality and PIN only."\n'
    'aaacf177-9918-489f-873c-7ecbba0b7117,address_stateOrRegion,Delhi/ncr,Uttar Pradesh,source_backfill,https://magicpin.in/Noida/Sector-104/Healthcare/Vishwasth-Clinic/store/97a203,https://nidhi.tourism.gov.in/home/classified?categoryCode=02&pageno=30&stateCode=,"Clinic listing matches Ground Floor, Homestop 104 opposite Pathways School. The government directory and postal snapshot place Noida Sector 104, PIN 201304, in Uttar Pradesh."\n'
    'bdc29cf4-75ba-4058-af10-2e58fc0d8885,address_stateOrRegion,Latur,Maharashtra,source_backfill,https://www.salunkecriticalcarehospital.com/contact.php,https://www.indiapost.gov.in/Financial/DOP_PDFFiles/PostOfficesOnline.pdf,The hospital identifies Salunke Complex opposite the bus stand in Nilanga 413521. India Post assigns the PIN to Maharashtra.\n'
    'c250d425-7c9b-4f30-8341-41f6047bd77c,address_stateOrRegion,Pune,Maharashtra,source_backfill,https://www.khadehospital.com/contact-us/,https://www.esic.gov.in/attachments/sstfile/f06d9c22f2371570140c2a990bbb08ed.pdf,"The official and ESIC addresses match Bija Complex, near the Chakan bus stand, Pune 410501, Maharashtra."\n'
    'c274a381-1d32-4d5c-8c5a-116ed6eb5253,address_stateOrRegion,Hoogly,West Bengal,source_backfill,https://www.wbpcb.gov.in/files/Fr-02-2025-02-31-42Unique%20number_HCF.pdf,,"The state register distinguishes the R. S. Verma Road, Uttarpara 712258 branch from the Chanditala branch."\n'
    'f81f97a0-9f91-4682-a285-47ba49df0daa,address_stateOrRegion,Up,Uttar Pradesh,source_backfill,https://www.lifetronhospital.com/info/contact-us,,"Hospital contact page matches O/PR-2, O Block, Kanpur 208016. Expand UP using the postal directory."\n'
    '0339d6ba-c0ed-4b7c-a15e-4077288dc082,address_stateOrRegion,South 24 Parganas,West Bengal,source_backfill,https://www.wbpcb.gov.in/files/Fr-02-2025-02-31-42Unique%20number_HCF.pdf,,"The state register lists Ashsheefa Hospital at Falta, South 24 Parganas, PIN 743504."\n'
    '118304aa-0118-42a2-8cb5-04b49ae73bfc,address_stateOrRegion,Thane,Maharashtra,source_backfill,https://kwa.kerala.gov.in/wp-content/uploads/2024/07/Employees-LIST-OF-NETWORKS-HOSPITALS_PAN-INDIA-_Ms-United-India-Insurance-Company.pdf,https://hrms.pnb.bank.in/pdf/NOTICE0111.pdf,"Both listings match Aakash Arcade near Bhanu Sagar Theatre on Dr Deepak Shetty Road and place it in Kalyan West, Maharashtra 421301."\n'
    '118304aa-0118-42a2-8cb5-04b49ae73bfc,address_city,Dombivli East,Kalyan,source_backfill,https://kwa.kerala.gov.in/wp-content/uploads/2024/07/Employees-LIST-OF-NETWORKS-HOSPITALS_PAN-INDIA-_Ms-United-India-Insurance-Company.pdf,https://hrms.pnb.bank.in/pdf/NOTICE0111.pdf,"Both listings match Aakash Arcade near Bhanu Sagar Theatre on Dr Deepak Shetty Road and place it in Kalyan West, Maharashtra 421301."\n'
    '38eee935-7af8-4af0-867a-210b03e6eaa6,address_stateOrRegion,Burhanpur,Madhya Pradesh,source_backfill,https://alliswellhospital.com/,https://portal.nabh.co/Documents/AccreditedList/Hospitals/H-2024-1410_5th%20Edition.pdf,"Facility and NABH sources match the Mohammadpur site near Macro Vision Academy, Burhanpur 450331."\n'
    '42e3862b-a20b-4eab-87c3-2d1f8f360518,address_stateOrRegion,Pimpri-chinchwad,Maharashtra,source_backfill,https://kwa.kerala.gov.in/wp-content/uploads/2024/07/PEN-LIST-OF-NETWORK-HOSPITALS_PAN-_INDIA_Ms-ORIENTAL-INSURANCE-COMPANY-LTD.pdf,,"The published insurer list identifies 7 Orange Hospital at 74 Pavana Nagar, Chinchwad, Maharashtra 411033."\n'
    '5accfa52-afb2-4bc2-8a65-23880b92104e,address_stateOrRegion,Kannur,Kerala,source_backfill,https://kwa.kerala.gov.in/wp-content/uploads/2024/07/Employees-LIST-OF-NETWORKS-HOSPITALS_PAN-INDIA-_Ms-United-India-Insurance-Company.pdf,https://kwa.kerala.gov.in/wp-content/uploads/2024/07/PEN-LIST-OF-NETWORK-HOSPITALS_PAN-_INDIA_Ms-ORIENTAL-INSURANCE-COMPANY-LTD.pdf,"Published insurer directories match Dhanalakshmi Hospital on Kannothumchal Road, Kannur, Kerala 670002."\n'
    '804cd688-8484-4c54-84ee-3ac6d1d27ab2,address_stateOrRegion,Bigbara,Uttarakhand,source_backfill,https://themedicityrudrapur.com/Packages.php,,"The hospital lists Teen Pani, Bigbara, Uttarakhand 263153."\n'
    'c2b79056-3772-459b-8693-8ed128e86e46,address_zipOrPostcode,517300,517325,source_backfill,https://swaasthyahospitals.com/contact/,https://satsang-foundation.org/initiatives/health-and-wellness/swaasthya-hospital,The hospital and its parent foundation identify the Nakkaladinne Madanapalle site. The facility lists 517325.\n'
    'c2c2f471-241c-489e-9d0d-e422fa66ff7f,address_zipOrPostcode,110007,131001,source_backfill,https://portal.nabh.co/Documents/AccreditedList/Hospitals/H-2021-0803_5th%20Edition.pdf,https://health.py.gov.in/sites/default/files/empanelledhospitals.pdf,"NABH certifies the Industrial Area, Rohtak Road branch at Sonipat, Haryana 131001. The old point still requires screening after the PIN correction."\n'
    'c2c2f471-241c-489e-9d0d-e422fa66ff7f,address_city,Delhi,Sonipat,source_backfill,https://portal.nabh.co/Documents/AccreditedList/Hospitals/H-2021-0803_5th%20Edition.pdf,https://health.py.gov.in/sites/default/files/empanelledhospitals.pdf,"NABH certifies the Industrial Area, Rohtak Road branch at Sonipat, Haryana 131001. The old point still requires screening after the PIN correction."\n'
    'd9178b81-b6a0-4558-a977-7760e67ba8c2,address_zipOrPostcode,131031,131001,source_backfill,https://portal.nabh.co/Documents/AccreditedList/Hospitals/H-2020-0694_5th%20Edition.pdf,https://www.fimssonipat.com/,"NABH identifies FIMS Hospital on Sonipat–Bahalgarh Road, Sonipat, Haryana 131001. The hospital identifies the same road."\n'
    'e5a024f8-154a-4bc1-b8ee-a352d6a72e1a,address_stateOrRegion,Dhule,Maharashtra,source_backfill,https://www.mospi.gov.in/sites/default/files/6ec_dirEst/ec6_Maharashtra.pdf,https://dhule.gov.in/en/std-pin-codes/,Economic Census lists Bafna Hospital in Dhule 424001. The government confirms the city PIN. This confirms the state only.\n'
    'f3d903e7-cc15-4a40-a944-25af1459e07f,address_zipOrPostcode,431027,411027,source_backfill,https://www.puneurologist.in/contact/,https://www.indiapost.gov.in/Financial/DOP_PDFFiles/PostOfficesOnline.pdf,"The practice address matches 201 Avenue Plaza above Reliance Digital, Pimple Saudagar, Pune 411027. India Post confirms 411027 in Maharashtra."\n'
    'fa93aa0d-a544-4657-b40e-35c5972d204c,address_stateOrRegion,Mohali,Punjab,source_backfill,https://www.holybasilmediclinic.com/contactus.php,https://dhsr.hp.gov.in/Files/List%20of%20Empanelled%20Hospitals%20under%20ESI%20Scheme%20HP%20as%20on%2027.10.2025.pdf,"Both sources match plots 139–141, Prime City, Landran Road, Kharar, Mohali, Punjab 140301."\n'
    '3057301e-4a0a-41c9-9d84-fdefbfedc7a9,address_zipOrPostcode,122022,122001,source_backfill,https://www.lifeaid.in/contact,,The facility matches door number 1097 near Unitech Cyberpark in Gurugram and gives 122001.\n'
    '5a6fe956-8f33-4fbe-b232-1de883afc999,address_zipOrPostcode,000000,440009,source_backfill_unlisted_pin,https://moil.nic.in/userfiles/NAGPUR_CITY_NETOWRK_HOSPITALS_13-07-2025%281%29.pdf,https://www.maharashtramedicalcouncil.in/Files/REGISTER_31122020.pdf,"The facility listing and medical council address match 20 Reshim Bagh, Umred Road, Nagpur. The PIN is missing from the supplied coordinate snapshot."\n'
    '612a341a-18d4-4cda-9052-c07c4e9aa4e4,address_stateOrRegion,Idukki,Kerala,source_backfill,https://sha.kerala.gov.in/?lang=en&page_id=573,,"The Kerala State Health Agency identifies the Murickassery, Idukki 685604 facility. The record also has an unmatched Karukutty fragment."\n'
    '63a98497-07ae-4af1-b0cc-7fd3cf561988,address_zipOrPostcode,39,411039,source_backfill,https://kwa.kerala.gov.in/wp-content/uploads/2024/07/Employees-LIST-OF-NETWORKS-HOSPITALS_PAN-INDIA-_Ms-United-India-Insurance-Company.pdf,,"The published insurer listing matches Pune–Nashik Road opposite PMT Chowk, Bhosari, Pune 411039."\n'
    '6439a466-5d68-46c3-a68c-b6d1532d00bc,address_zipOrPostcode,17,700017,source_backfill,https://finance.wb.gov.in/writereaddata/196-F%28MED%29-WB.pdf,https://www.wbpcb.gov.in/files/Fr-02-2025-02-31-42Unique%20number_HCF.pdf,"The finance department notice and state regulator match Dr S S Chatterjee Heart Centre, 11/5 Biresh Guha Street. The finance notice gives Kolkata 700017."\n'
    '6a46bfc2-421f-44b5-be16-82aae258acca,address_stateOrRegion,Indore,Madhya Pradesh,source_backfill,https://gitabhawanhospital.org/,https://mppost.gov.in/SubPostOffice.aspx,"The hospital identifies Indore, Madhya Pradesh. India Post places Manoramaganj 452001 in Indore."\n'
    '71fe0119-a6c0-4497-aacb-749c11e42864,address_stateOrRegion,Thrissur,Kerala,source_backfill,https://www.dayaroyalhospital.com/contact-us,https://norkaroots.kerala.gov.in/hospitallist/Kerala_Hospital_data.pdf,"The hospital and public directory identify Guruvayur Road, Kunnamkulam, Thrissur, Kerala 680503."\n'
    '896f0b3e-2a56-4f89-b0f1-cdf50773a016,address_stateOrRegion,Cuddalore,Tamil Nadu,source_backfill,https://kwa.kerala.gov.in/wp-content/uploads/2024/07/Employees-LIST-OF-NETWORKS-HOSPITALS_PAN-INDIA-_Ms-United-India-Insurance-Company.pdf,,"The published insurer listing matches 9 Imperial Road, Cuddalore, Tamil Nadu 607002."\n'
    'd0c33da2-de94-4673-ae16-899ed4a8a486,address_zipOrPostcode,85,110085,source_backfill,https://www.vaidmishraji.in/sinusitis/,https://www.indiapost.gov.in/Financial/DOP_PDFFiles/PostOfficesOnline.pdf,"The clinic lists E-16/30, Sector 8, Rohini, Delhi with shorthand 85. India Post corroborates Rohini 110085."\n'
    'da424446-f4fe-4e4b-ac78-795b3b1a8786,address_stateOrRegion,Mh,Maharashtra,source_backfill,https://charity.maharashtra.gov.in/CharityHospitals-en-US?id=25,https://charity.maharashtra.gov.in/Portals/0/Files/Hospital_scheme_document.pdf,"Maharashtra\'s charity register identifies the hospital complex at 124 Anandrishiji Marg, Burudgaon Road, Ahmednagar 414001. Expand the state abbreviation Mh."\n'
    'e61063fc-06af-4cd8-925d-ded933eebd39,address_zipOrPostcode,000000,221002,source_backfill,https://d3u7ubx0okog7j.cloudfront.net/documents/Consolidated_List_of_Empaneled_Hospitals_of_Coal_India_Limited_as_on_15.02.2024.pdf,,"The Coal India empanelment document matches S-17/329, Maldahiya Cantt opposite Roadways Workshop, Varanasi 221002."\n'
    'eac16a4a-5a75-4524-9abe-98e8397128d3,address_stateOrRegion,Uttar Dinajpur,West Bengal,source_backfill,https://cdnbbsr.s3waas.gov.in/s3aae8d1e00b15a30e5901227e97ffbef7/uploads/2025/08/2025080193.pdf,,"Government institution list identifies Dr B C Roy Sarani, Raiganj, Uttar Dinajpur, West Bengal 733134."\n'
    'f820f89d-6ef2-44d7-bb58-ab2da0300c7b,address_stateOrRegion,Thane,Maharashtra,source_backfill,https://bilal-hospital.com/contact,https://kwa.kerala.gov.in/wp-content/uploads/2024/07/Employees-LIST-OF-NETWORKS-HOSPITALS_PAN-INDIA-_Ms-United-India-Insurance-Company.pdf,"Both sources match Royal Garden, A Wing near Shimla Park, Mumbra, Thane, Maharashtra 400612."\n'
    'fac0cc70-81a9-462d-bced-5ca40257b027,latitude,59.949684143066406,9.259299,source_backfill,https://smsh.in/,"https://www.google.com/maps/place/Sanjivani%2BHospital/@9.259299,76.5961806,17z/data=!3m1!4b1!4m6!3m5!1s0x3b0619dd233ad893:0x4c386f24b4330f31!8m2!3d9.259299!4d76.5961806!16s%2Fg%2F11f7r0lkj1?entry=ttu","The hospital\'s Our Location link selects a named facility marker for its Kollakadavu PO, Chengannur, Kerala 690509 campus. This is a campus map marker."\n'
    'fac0cc70-81a9-462d-bced-5ca40257b027,longitude,-38.26263427734375,76.5961806,source_backfill,https://smsh.in/,"https://www.google.com/maps/place/Sanjivani%2BHospital/@9.259299,76.5961806,17z/data=!3m1!4b1!4m6!3m5!1s0x3b0619dd233ad893:0x4c386f24b4330f31!8m2!3d9.259299!4d76.5961806!16s%2Fg%2F11f7r0lkj1?entry=ttu","The hospital\'s Our Location link selects a named facility marker for its Kollakadavu PO, Chengannur, Kerala 690509 campus. This is a campus map marker."\n'
    'fac0cc70-81a9-462d-bced-5ca40257b027,coordinates,"{""coordinates"":[-38.26263427734375,59.949684143066406],""type"":""Point""}","{""coordinates"":[76.5961806,9.259299],""type"":""Point""}",source_backfill,https://smsh.in/,"https://www.google.com/maps/place/Sanjivani%2BHospital/@9.259299,76.5961806,17z/data=!3m1!4b1!4m6!3m5!1s0x3b0619dd233ad893:0x4c386f24b4330f31!8m2!3d9.259299!4d76.5961806!16s%2Fg%2F11f7r0lkj1?entry=ttu","The hospital\'s Our Location link selects a named facility marker for its Kollakadavu PO, Chengannur, Kerala 690509 campus. This is a campus map marker."\n'
    '439fc924-7e86-4631-8723-374a70e61b20,address_stateOrRegion,Hyderabad,Telangana,source_backfill,https://www.zoihospitals.com/contacts-us/,https://www.zoihospitals.com/zoi-attapur/,"Official Attapur page confirms the branch, Telangana and 500048. The source reports Pillar 145 while the input reports 147. The state is supported."\n'
    '6094934d-97b6-4598-87d1-70d31a4a9c94,address_zipOrPostcode,35,600035,source_backfill,https://portal.nabh.co/Documents/AccreditedList/Entry-Level%20Hospitals/PEH-2018-0595.pdf,https://health.py.gov.in/sites/default/files/empanelledhospitals.pdf,"NABH matches 36-A Chamiers Road, Nandanam, Chennai 600035. The older directory corroborates the site."\n'
    '7d1d6e4e-43b4-44a2-a21a-60d67417d789,address_stateOrRegion,Andhra Pradesh,Telangana,source_backfill,https://in.linkedin.com/company/hypro-diagnostics-centre,https://tgct.gov.in/tgportal/aboutus/KnowYourJurDtls.aspx,"Facility-operated profile matches door 1198, Road 59, Jubilee Hills, Telangana 500033. Government jurisdiction and postal evidence corroborate the locality."\n'
    '851a4aba-ea01-4016-8f8f-f972ca4d448d,address_stateOrRegion,Palghar,Maharashtra,source_backfill,https://radianthospitalvirar.com/contact-us.html,https://www.bharatibiz.com/en/radiant-hospital_228-083292-32790,"The hospital matches Sheetal Anjali Building near D Mart, Manvelpada, Virar East 401305. The independent listing and postal reference place it in Maharashtra."\n'
    '0201050b-de4b-46ab-9425-68147e690173,address_line1,"57-7-25, New Postal Colony Road, Near Sonovision","57-7-23/1, New Postal Road, Near Sonovision",source_backfill,https://www.vedakidneyhospital.in/about.html,https://www.dnb.com/business-directory/company-profiles/veda-hospital--kidney-centre%C2%A0.3b24e8d89fb3f867a3f3cf1195ec4c65,"The clinic name, Patamata locality, Sonovision landmark and same public telephone identify the branch. The facility page and D&B list door 57-7-23/1. D&B supplies PIN 520010."\n'
    '0201050b-de4b-46ab-9425-68147e690173,address_zipOrPostcode,10,520010,source_backfill,https://www.vedakidneyhospital.in/about.html,https://www.dnb.com/business-directory/company-profiles/veda-hospital--kidney-centre%C2%A0.3b24e8d89fb3f867a3f3cf1195ec4c65,"The clinic name, Patamata locality, Sonovision landmark and same public telephone identify the branch. The facility page and D&B list door 57-7-23/1. D&B supplies PIN 520010."\n'
)


In [ ]:
WEBSITE_DECISIONS_JSON = (
    '[\n'
    '  {\n'
    '    "id": "ac4c39ce-8c64-4bab-a4db-1b3dca1ea817",\n'
    '    "facility": "Jyoti Hospital",\n'
    '    "field": "websites",\n'
    '    "original_value": "netrameeye hospital.com",\n'
    '    "retained_value": "https://jyotieyehospital.org/",\n'
    '    "source_url": "https://jyotieyehospital.org/",\n'
    '    "note": "Retain the Jyoti Eye Hospital site for Visnagar. The similarly named Netram domain is associated with a different provider.",\n'
    '    "corroborating_url": "https://happyinsurance.in/health-insurance/visnagar-gujarat/"\n'
    '  },\n'
    '  {\n'
    '    "id": "71c0cdb6-b7a5-4c05-b8ac-b0beae74d0d9",\n'
    '    "facility": "Jain ENT Hospital",\n'
    '    "field": "websites",\n'
    '    "original_value": "ja jaipur? ja jaipur ent hospital official site?",\n'
    '    "retained_value": "https://jainenthospital.org/",\n'
    '    "source_url": "https://jainenthospital.org/contact-us/",\n'
    '    "note": "The contact page identifies Jain ENT Hospital in Jaipur."\n'
    '  },\n'
    '  {\n'
    '    "id": "71c0cdb6-b7a5-4c05-b8ac-b0beae74d0d9",\n'
    '    "facility": "Jain ENT Hospital",\n'
    '    "field": "websites",\n'
    '    "original_value": "ja jaipur?",\n'
    '    "retained_value": "https://jainenthospital.org/",\n'
    '    "source_url": "https://jainenthospital.org/contact-us/",\n'
    '    "note": "The contact page identifies Jain ENT Hospital in Jaipur."\n'
    '  },\n'
    '  {\n'
    '    "id": "dc97c031-cec5-48eb-a1c1-d4de6520eb63",\n'
    '    "facility": "Capitol Hospital",\n'
    '    "field": "websites",\n'
    '    "original_value": "capitolhospital",\n'
    '    "retained_value": "https://www.capitolhospital.com/",\n'
    '    "source_url": "https://www.capitolhospital.com/page/contact-us",\n'
    '    "note": "The hospital contact page identifies the matching site in Jalandhar."\n'
    '  },\n'
    '  {\n'
    '    "id": "f947f247-8b7a-4ea0-a6b6-92a18e13e025",\n'
    '    "facility": "Kansal Dental Clinic and Orthodontic Centre",\n'
    '    "field": "websites",\n'
    '    "original_value": "kansaldentalclinic",\n'
    '    "retained_value": "https://kansalorthodonticcentre.in/",\n'
    '    "source_url": "https://kansalorthodonticcentre.in/contact/",\n'
    '    "note": "The clinic contact page identifies the matching practice in Meerut."\n'
    '  },\n'
    '  {\n'
    '    "id": "d0f3824e-0da4-431c-88d1-cdffb4d9e383",\n'
    '    "facility": "Satyam Clinical Laboratory",\n'
    '    "field": "websites",\n'
    '    "original_value": "sat yam lab?/.",\n'
    '    "retained_value": "https://www.satyamlab.com/",\n'
    '    "source_url": "https://www.satyamlab.com/",\n'
    '    "note": "The laboratory site identifies Satyam Clinical Laboratory in Chandigarh."\n'
    '  },\n'
    '  {\n'
    '    "id": "fccc2b9a-9df1-4883-a05e-d35a6de669f7",\n'
    '    "facility": "Green City Hospital",\n'
    '    "field": "websites",\n'
    '    "original_value": "greencityhospitalbhopal.com? to be replaced",\n'
    '    "retained_value": "https://www.greencityhospitalbhopal.com/",\n'
    '    "source_url": "https://www.greencityhospitalbhopal.com/contact-us/",\n'
    '    "note": "The contact page identifies the matching Green City Hospital in Bhopal."\n'
    '  },\n'
    '  {\n'
    '    "id": "56c32a1e-9110-4cc9-8061-17b807194366",\n'
    '    "facility": "Anu Specialty Hospital",\n'
    '    "field": "websites",\n'
    '    "original_value": "anu_multispeciality_hospital",\n'
    '    "retained_value": "https://www.anuhospital.in/",\n'
    '    "source_url": "https://www.anuhospital.in/",\n'
    '    "note": "The hospital site identifies the matching Anu hospital in Thanjavur."\n'
    '  },\n'
    '  {\n'
    '    "id": "d25b05cb-f9f6-43d4-8506-3e872605fc2c",\n'
    '    "facility": "Sarojini Dental Hospital",\n'
    '    "field": "websites",\n'
    '    "original_value": "sarojinidental hospital.com",\n'
    '    "retained_value": "https://www.sarojinidental.com/",\n'
    '    "source_url": "https://www.sarojinidental.com/contact",\n'
    '    "note": "The contact evidence matches the Sarojini Dental site in Secunderabad."\n'
    '  },\n'
    '  {\n'
    '    "id": "f5787e6a-9015-419a-b6df-65d5d0c7dcb1",\n'
    '    "facility": "Dr. Varshali\'s Gynecology Clinic",\n'
    '    "field": "websites",\n'
    '    "original_value": "https://www.drvarshaliclinic.com/about-dr-varshali-mali-gynecologist-a\\u001e\\u0016",\n'
    '    "retained_value": "https://www.drvarshaliclinic.com/about-dr-varshali-mali-gynecologist-and-pregnancy-doctor",\n'
    '    "source_url": "https://www.drvarshaliclinic.com/about-dr-varshali-mali-gynecologist-and-pregnancy-doctor",\n'
    '    "note": "The full provider profile identifies Dr. Varshali Mali. The malformed value ends with two control characters."\n'
    '  },\n'
    '  {\n'
    '    "id": "287913e3-162c-45d5-873a-eada15f8591b",\n'
    '    "facility": "Dr. Manjari Ultrasound",\n'
    '    "field": "websites",\n'
    '    "original_value": "dr.manjari_ultrasound.center",\n'
    '    "retained_value": "https://drmanjariultrasound.com/",\n'
    '    "source_url": "https://drmanjariultrasound.com/contact-us.php",\n'
    '    "note": "The contact page identifies the matching ultrasound centre in Noida."\n'
    '  },\n'
    '  {\n'
    '    "id": "8735c338-9b18-43d1-a4da-1e835fd1559a",\n'
    '    "facility": "Comfort Multi Speciality Hospital",\n'
    '    "field": "officialWebsite",\n'
    '    "original_value": "com forthospitals.com",\n'
    '    "source_url": "https://kwa.kerala.gov.in/wp-content/uploads/2024/07/PEN-LIST-OF-NETWORK-HOSPITALS_PAN-_INDIA_Ms-ORIENTAL-INSURANCE-COMPANY-LTD.pdf",\n'
    '    "note": "The directory supports the Bangalore facility identity. Ownership of the candidate domain could not be verified."\n'
    '  },\n'
    '  {\n'
    '    "id": "76a3a7f6-d5f5-4dd8-a6c9-1fa614928e79",\n'
    '    "facility": "ESI Hospital",\n'
    '    "field": "officialWebsite",\n'
    '    "original_value": "[\\"esic.nic.in\\"]",\n'
    '    "source_url": "https://ims.kerala.gov.in/en/node/28",\n'
    '    "note": "The government page identifies a site in Olarikkara, Thrissur. The input mixes Perumbavoor with PIN 691536. A branch website could not be assigned."\n'
    '  },\n'
    '  {\n'
    '    "id": "7b2e1044-8170-42b1-ad7f-f6501255ba7e",\n'
    '    "facility": "Lifeline Clinic & Diagnostics",\n'
    '    "field": "officialWebsite",\n'
    '    "original_value": "lifelineclinicnd diagnostics.com",\n'
    '    "source_url": "https://lifelineclinicandiagnostics.com/",\n'
    '    "note": "The old candidate could not be reached. The candidate active site lists Govind Nagar, while the input lists Pathardi Phata."\n'
    '  },\n'
    '  {\n'
    '    "id": "39c6f2b8-806a-499c-9fac-a918e52cd191",\n'
    '    "facility": "Kasturba Hospital",\n'
    '    "field": "websites",\n'
    '    "original_value": "kasturbamultispeciality",\n'
    '    "source_url": "https://www.signalhire.com/companies/kasturba-hospital-tumkur",\n'
    '    "note": "The record mixes Tumkur and Manipal sources. The candidate domain could not be tied to the recorded branch."\n'
    '  },\n'
    '  {\n'
    '    "id": "2d65a95f-4842-4d1d-9ddd-fc94ddb153c3",\n'
    '    "facility": "Jivandeep Hospital",\n'
    '    "field": "websites",\n'
    '    "original_value": "jivandeep hospital.com",\n'
    '    "source_url": "https://in.linkedin.com/company/jivandeephospital",\n'
    '    "note": "The candidate site is associated with Dholka, while the input identifies Santrampur."\n'
    '  },\n'
    '  {\n'
    '    "id": "50fd571f-6d38-43a7-a8a0-b9a651b25f37",\n'
    '    "facility": "Smile N Shine",\n'
    '    "field": "websites",\n'
    '    "original_value": "http://smile-n-shine.in/dental clinic",\n'
    '    "source_url": "https://chotu.com/local/dental-clinics-in-khardaha-kolkata-west-bengal-india/ct-18418140/",\n'
    '    "note": "The directory supports the Barrackpore clinic identity. A replacement website could not be verified."\n'
    '  },\n'
    '  {\n'
    '    "id": "67dc110e-5e33-472b-89ad-984064802d21",\n'
    '    "facility": "Ethos Body and Mind Clinic",\n'
    '    "field": "websites",\n'
    '    "original_value": "ethos hypnotherapyclinic?",\n'
    '    "source_url": "https://www.ethoshypnotherapyclinic.com/feedback/index.html",\n'
    '    "note": "The indexed page matched the contact details, but the live candidate returned HTTP 502 during review."\n'
    '  },\n'
    '  {\n'
    '    "id": "6d50cfb6-89dc-438c-801f-c590c0c81335",\n'
    '    "facility": "Vighneshwar Ayurved Hospitals",\n'
    '    "field": "websites",\n'
    '    "original_value": "vighneshwarayurvedhospitals",\n'
    '    "source_url": "https://www.practo.com/sangli/clinic/vighneshwar-ayurved-hospitals-sangli-city",\n'
    '    "note": "The directory identifies the Sangli clinic. An official replacement site could not be verified."\n'
    '  },\n'
    '  {\n'
    '    "id": "836a84fc-cdf2-4496-91e1-0accd20ab2d1",\n'
    '    "facility": "Promise Healthcare Hospital",\n'
    '    "field": "websites",\n'
    '    "original_value": "promisehealthcareranchi",\n'
    '    "source_url": "https://yappe.in/jharkhand/ranchi/dr-deepak-verma/607516",\n'
    '    "note": "A third-party listing links the candidate to the Ranchi facility. The candidate was unavailable during review."\n'
    '  }\n'
    ']\n'
)


In [ ]:
FACILITY_NAME_DECISIONS_CSV = (
    'unique_id,expected_name,cleaned_name,source_url,note\n'
    '8c4ad55c-ad0b-40f7-be7e-d9616565cf6e,"The Corporate Hospital - Batala Road, Amritsar-143001, Punjab, India",The Corporate Hospital,,The title suffix repeats the structured street and PIN.\n'
    '3496cf37-2efe-4b43-816c-c0a62712baa5,"Redcliffe Labs - Collection Center Ground Floor, Jitan Chowk, Dighi Kala West, Near Station Road Plat Form No 5 Hajipur, Bihar - 844101",Redcliffe Labs (Hajipur Collection Center),,Preserve the collection center designation and the recorded city.\n'
    '8354ebb1-e2c1-4060-93cd-94c84c5f2ab4,"Nagpal Nursing & Maternity Home - Kurukshetra Sector 17, Kurukshetra Red Road, Sector 17, Kurukshetra-136118, Haryana",Nagpal Nursing & Maternity Home,,The title suffix repeats the structured street and PIN.\n'
    '8806e733-3f36-46dc-bf99-8b824b38a3dd,"Pooja Surgical Hospital - Vanshiro House, 30 Karan Para, 2nd Floor, Karansinhji Main Road, Rajkot-360001, Gujarat",Pooja Surgical Hospital,,The title suffix repeats the structured street and PIN.\n'
    '1771720d-5f9d-4ad2-a080-b1e007cb06cf,"Tathagat Heart Hospital - 31 Crescent Road, 1st Floor, Mallige Premises, Bengaluru, Karnataka 560001",Tathagat Heart Hospital,,The title suffix repeats the structured street and PIN.\n'
    '2045489d-75e9-4aa6-8d6a-02d877491fc3,"Esthetica International Clinique - 95P First Floor, Sector 47, Gurgaon, Haryana - 122018, India, South City ii, Gurugram",Esthetica International Clinique,,The title suffix repeats the structured street and PIN.\n'
    '8146f2e6-3f68-42e5-a54b-d8281aa05081,"Sparsh Hospital - Plot No 141, Sai Arcade, Mission Compound, Line Ali, Opposite Orion Mall, Panvel, Navi Mumbai, Maharashtra 410206",Sparsh Hospital Panvel,https://sparshhospital.in/contact-sparsh-hospital-panvel/,The hospital contact page uses this branch name.\n'
    '987f9fcd-db93-4abe-afec-1b07f94698da,"Dr. Mithari’s Rajmatiraj Pain Clinic & Homeopathic Research Center - 311, E-Ward, Near Hotel Maratha Regency, New Shahupuri, Kolhapur- 416001",Dr. Mithari’s Rajmatiraj Pain Clinic & Homeopathic Research Center,,The title suffix repeats the structured street and PIN.\n'
    'b39153f3-664a-45c4-9862-4dec3e19fab3,"Shyam Nidan Clinic And Shyam Diagnostics - HS-1/45, Sector A, Jankipuram Sitapur Road Yojna, Lucknow-226021, Uttar Pradesh, India",Shyam Nidan Clinic And Shyam Diagnostics,,The title suffix repeats the structured street and PIN.\n'
    'e4e3dd4b-f20a-4d59-b68c-e363da6248f5,"Jordan Skin Clinic - 1st Floor, MQ Splendor, Opp Pillar No 209, Inner Ring Road Upperpally, Rajendra Nagar Mandal, Hyderabad, Telangana, India - 500030",Jordan Skin Clinic,,The title suffix repeats the structured street and PIN.\n'
    '1efb1d48-5b53-4d61-bd68-3f977b727c42,"Opal Care Clinic - Shop No 29 Lower Ground Floor, Mahagun Mart, 2, 11th Ave, opp. to Greater Noida West Road, Gaur City 2, Greater Noida, Ghaziabad, Uttar Pradesh 201309",Opal Care Clinic,,The title suffix repeats the structured street and PIN.\n'
    'a169589d-5bec-4da8-b07a-5919ed0aafbd,"Hope Centre For Speech Therapy And Autism - Building No Gate No.577, opp. Vardan Hospital, near Bus Stand, Mota Singh Nagar, Jalandhar, Punjab 144022",Hope Centre For Speech Therapy And Autism,,The title suffix repeats the structured street and PIN.\n'
    'b572ed14-25bb-4cdb-9da1-497ea512e5c5,"Krishna Rajendra Hospital - KR Hospital, Irwin Rd, Devaraja Mohalla, Yadavagiri, Mysuru, Karnataka 570001",Krishna Rajendra Hospital (KR Hospital),,Preserve the hospital alias in the source title.\n'
    'd0cb9f92-f2a8-4b11-a4df-023b1decbeca,"Paras Dental Care Clinic 2 - 505, Laxman Path, opposite Stylush salon, Vivek Vihar, Shyam Nagar, Jaipur, Rajasthan 302019",Paras Dental Care Clinic 2,,The title suffix repeats the structured street and PIN.\n'
    'dca514ea-e154-47ce-8510-149ce2d344a8,"EHCC Hospital - 3A Jagatpura Road, Near Jawahar Circle, Jaipur, Rajasthan 302017",EHCC Hospital,,The title suffix repeats the structured street and PIN.\n'
    '0bc38214-36fd-48a0-9fac-e0523d529a26,"Navjeevan Heart and Women\'s Hospital - Near Horizon Hotel, Central Excise Road(GST Bhawan) GIDC Ankleshwar, 393002",Navjeevan Heart and Women\'s Hospital,,The title suffix repeats the structured street and PIN.\n'
    '255940ad-20c1-402b-86c0-0bf90d40f851,"Health Quest Imaging and Diagnostics - Near Manesar Bus Stand, Ground Floor, Shanti Complex, Manesar, Gurugram, Haryana 122052",Health Quest Imaging and Diagnostics,,The title suffix repeats the structured street and PIN.\n'
    '44aacde8-ff8b-4891-b46a-7c6f21821ed9,"Nucleus The Diagnostic Centre - B-Wing, 1st Floor, Tulsi Samarth Building, Opp. Gauripada Talav, Kalyan (W), Maharashtra - 421203",Nucleus The Diagnostic Centre,,The title suffix repeats the structured street and PIN.\n'
    '58d82919-60f7-4a74-a45f-0246f13d1b49,"Samriddhi Hospital and Fertility Center - Plot No. 5, Mata Amritanandamayi Marg, opposite Indian Oil Petrol Pump, Sector 88, Faridabad, Haryana - 121002",Samriddhi Hospital and Fertility Center,,The title suffix repeats the structured street and PIN.\n'
    '7eddf7aa-3fd9-42b2-92e9-04cd81a49057,"Aarna Diagnostic Centre - Main Centre, 86, Kargi - Patel Nagar Bypass, near Palm City, Adersh Vihar, Brahmanwala, Dehradun, Uttarakhand 248001",Aarna Diagnostic Centre (Main Centre),,Preserve the Main Centre designation.\n'
    'c10d22d8-aba2-4b43-ab67-cfcebc3acd81,"Kandamkulathy Super Speciality Ayurveda Hospital - Burma Road Junction, Kumarapuram Thiruvananthapuram, Kerala - 695011",Kandamkulathy Super Speciality Ayurveda Hospital,,The title suffix repeats the structured street and PIN.\n'
    'd829730f-417d-4d8e-974f-179334e202b4,"Grace Homoeo Care - 40/3-4 GP Complex, INTUC Colony Main Road Corner, Anna Main Street, Opp to Guru Theater, Bethaniapuram, Madurai-625016",Grace Homoeo Care,,The title suffix repeats the structured street and PIN.\n'
    '0655e834-cee2-430c-bd36-559d5e33dfe2,"Dr Rohit Nalavade\'s Clinic - Shop No 4, Umashakti Building, opposite Samarth gym, near Dayanand Anglo Vedic School, Sector-9, New Panvel East, Navi Mumbai, Maharashtra 410206",Dr Rohit Nalavade\'s Clinic,,The title suffix repeats the structured street and PIN.\n'
    '0f355580-dbcf-4f90-93c9-6bdb0fe4274b,"Trishaan Ayurvedic Health Clinic - Kiran Height Complex, Opposite of 218 Bus Stand, Kulpi Road, Baruipur, Kolkata-700144",Trishaan Ayurvedic Health Clinic,,The title suffix repeats the structured street and PIN.\n'
    '1d3f5f2e-e990-4bd8-abef-bd6c2b2b8d84,"Dr. Jairaj\'s Hospital - 2nd Floor, Om Chanakya Complex, Sector 6, CBD Belapur, Navi Mumbai, Maharashtra 400614",Dr. Jairaj\'s Hospital,,The title suffix repeats the structured street and PIN.\n'
    '3e8ed81c-e238-4b31-9dc8-195c3edcb63a,"Hearing Aid Center Delhi - No.143, Rishabh Vihar, Near Cross River Mall, Karkardooma, Opposite Aditya Mall, Delhi - 110092",Hearing Aid Center Delhi,,The title suffix repeats the structured street and PIN.\n'
    '45f9d3e3-b9a4-41cb-9de0-5ef799c42a42,"V Clear Aligners - Plot No 232, Lane between Centro and Reliance Trends, Near South India Shopping Mall, Mythri Nagar, Madinaguda, Hafeezpet, Hyderabad, Telangana 500049",V Clear Aligners,,The title suffix repeats the structured street and PIN.\n'
    '92f8389f-d6b0-400d-a4f2-a589b6186e48,"Dr. Nikhil\'s Homoeopathy - Namdev Road, Opposite Street No. 3 Near Gurudwara Sahib, Bathinda, Punjab (India) – 151001",Dr. Nikhil\'s Homoeopathy,,The title suffix repeats the structured street and PIN.\n'
    '9aaf086d-b927-4c98-966d-4f817f78ad31,"Dr. Dnyaneshwar Patil Mds Endodontist (Pro Dental Care) - 2nd Floor, Vision Galleria, Kunal Icon Road, Pimple Saudagar, Pune-411027, Maharashtra",Dr. Dnyaneshwar Patil Mds Endodontist (Pro Dental Care),,The title suffix repeats the structured street and PIN.\n'
    '9b9e2501-dd9f-4f35-94cf-ce4593d9e661,"Samved IVF - A3 Bhagwati Park Society, Opp. Tube Company, Old Padra Road, Vadodara, Gujarat 390020",Samved IVF,,The title suffix repeats the structured street and PIN.\n'
    '9e999deb-15d7-47ab-a0f7-57b2175cbcc3,"Agilus Diagnostics - No 735/5Main Road BuraiBurari New Delhi, Delhi - 110084",Agilus Diagnostics (New Delhi),,Use the recorded city to distinguish this branch.\n'
    'a36c6eec-79f4-470d-9614-467c12175466,"Agilus Diagnostics - Shop No 6, Double Story Booth No 6B Block Market Amritsar, Punjab - 143001",Agilus Diagnostics (Amritsar),,Use the recorded city to distinguish this branch.\n'
    'a37c0737-e643-4a78-b103-4eb421a6176e,"Mittal Diagnostic & Research Centre - Opp Eden Garden Banquet Hall, Kishanpur Tiraha, Ramghat Road Aligarh, Uttar Pradesh -202001 India",Mittal Diagnostic & Research Centre,,The title suffix repeats the structured street and PIN.\n'
    'af24376e-c5ca-4133-b75b-2bb44c55ccbb,"Nirali Hospital - A M Naik Healthcare Complex, Grid Road, Navsari, Gujarat - 396463, India",Nirali Hospital,,The title suffix repeats the structured street and PIN.\n'
    'b6b1aa24-78a7-4721-8a50-508b5bdd298b,"Dr Narkhede Ayurved & Panchkarm Clinic - Flat No 03,Ground Floor, C1 Aster, Sai Baba Vihar Complex, Ghodbunder Rd, near Muchhala Polytechnic College, Anand Nagar, Thane, Maharashtra 400615",Dr Narkhede Ayurved & Panchkarm Clinic,,The title suffix repeats the structured street and PIN.\n'
    'bda4beeb-d7e2-4a4c-94c9-01027cf2f2f7,"Dant Suraksha Dental Care - Navin Shahdara, Room No. 109, First Floor, L-1, Navin Shahdara, Shahdara, New Delhi, Delhi 110032",Dant Suraksha Dental Care (Navin Shahdara),,Preserve the recorded locality as the branch label.\n'
    'ef2331d9-201b-4f84-88f6-8575c5bc6939,"Maurya Dental Clinic - Building no. 161, Rd Number 2, Sri Krishna Nagar, Patna, Bihar 800001",Maurya Dental Clinic,,The title suffix repeats the structured street and PIN.\n'
    '1c5d4d78-c780-4b1a-875b-32280d82a84e,"Jain Dental Centre - F-19, Near Happy English School, Mandir Marg, Krishna Nagar, Delhi – 110051",Jain Dental Centre,,The title suffix repeats the structured street and PIN.\n'
    '3ad767f9-e38e-4280-9ecd-02cd308bd650,"Ace Dental Care - Sco No, 1, World One High Street, Sector 115, Landran Road, Near Hammer Gym, Sector 115, Mohali-140307, Punjab",Ace Dental Care,,The title suffix repeats the structured street and PIN.\n'
    '463c5cd0-46ba-428b-b395-dc95eb01b6f4,"Pathkind Labs - Joshi Hospital, Kapurthala Chowk, Durga Colony, Jalandhar, Punjab 144008","Pathkind Labs (Joshi Hospital, Jalandhar)",,Preserve the location identifier and city recorded for this site.\n'
    '5317561a-d29f-4f12-a929-e6d2c7a41b09,"{""name"":""St. Fatima Dental Clinic - 75, Rajpur Road, Dehradun (U.K.) 248001""}",St. Fatima Dental Clinic,,Read the name inside the JSON wrapper and remove the repeated street and PIN.\n'
    '531a29cc-4b07-495b-9bdb-daa7ead92d7e,"Medicity Speciality Laboratory - 12 No. Indra Biswas Road, Belgachia, Near Belgachia Post Office, Kolkata - 700037",Medicity Speciality Laboratory,,The title suffix repeats the structured street and PIN.\n'
    '59e6b2c9-eba8-41b0-a8ca-80a73f678bc1,"Renu Vision Care - Sco 18, Ground Floor, Lohgarh Road Zirkapur, Mohali-140413, Punjab, India",Renu Vision Care,,The title suffix repeats the structured street and PIN.\n'
    '74f83327-0fc5-4e81-82e4-ae094092d649,"General Diagnostics International - RPT House, Plot 6, Sector 24, Turbhe, Navi Mumbai, Maharashtra 400703",General Diagnostics International,,The title suffix repeats the structured street and PIN.\n'
    '8556c4f8-a814-44be-a2d4-3bf4b530b10d,"Diabetes and Thyroid Care - 2nd Floor, Synergy Healthcare, Swastik Hi-Point Gloria, Opp. Harmony Residences, Ovala, Ghodbunder Road, Owale, Thane West, Thane, Maharashtra 400615",Diabetes and Thyroid Care,,The title suffix repeats the structured street and PIN.\n'
    '8f63e350-3067-4734-8934-7a8a567da287,"Dr. Raj Kumar Deepak (Liver And Gastro Clinic) - Arya Samaj Chowk, Chhatauni Amar, Motihari-845401, Bihar",Dr. Raj Kumar Deepak (Liver And Gastro Clinic),,The title suffix repeats the structured street and PIN.\n'
)


def read_name_decisions():
    decisions = {}
    with StringIO(FACILITY_NAME_DECISIONS_CSV, newline="") as file:
        for decision in csv.DictReader(file):
            uid = decision["unique_id"]
            cleaned_name = decision["cleaned_name"].strip()
            if uid in decisions:
                raise ValueError(f"Duplicate facility name decision: {uid}")
            if not cleaned_name or cleaned_name == decision["expected_name"]:
                raise ValueError(f"Invalid facility name decision: {uid}")
            decision["cleaned_name"] = cleaned_name
            decisions[uid] = decision
    return decisions


## Silver · Clean and index the India Post directory

Keep one copy of exact duplicate office rows, validate PINs, correct reviewed office locations, and flag unusable coordinate pairs. Preserve distinct candidate states, districts, and reference coordinates for each PIN.


In [ ]:
def index_post(csv_path, dest):
    fields = ("pincode", "post_office_count", "state_candidates",
              "state_district_candidates", "valid_coordinate_count",
              "state_ambiguous", "district_ambiguous")
    counts = Counter()
    groups = defaultdict(lambda: {"rows": 0, "states": set(), "districts": set(), "points": set()})
    seen = set()
    removed = []
    with StringIO(POSTAL_SOURCE_DECISIONS_CSV, newline="") as f:
        overrides = {(r["pincode"], r["officename"]): r for r in csv.DictReader(f)}
    applied = set()
    with csv_path.open(encoding="utf-8-sig", newline="") as source:
        reader = csv.DictReader(source)
        with (dest/"pincode_offices_clean.csv").open("w", encoding="utf-8-sig", newline="") as output:
            writer = csv.writer(output)
            writer.writerow([*reader.fieldnames, "source_record", "state_key", "district_key",
                             "coordinate_status"])
            for number, row in enumerate(reader, 1):
                counts["input_rows"] += 1
                key = tuple(row.values())
                if key in seen:
                    counts["exact_duplicates_removed"] += 1
                    removed.append(("postal", number, "", "", row["pincode"], "",
                                    "remove_row", "exact_duplicate", "", "", ""))
                    continue
                seen.add(key)
                pin = re.sub(r"\s+", "", row["pincode"])
                if not PIN.fullmatch(pin):
                    counts["invalid_pins_removed"] += 1
                    removed.append(("postal", number, "", "pincode", row["pincode"], "",
                                    "remove_row", "invalid_pin", "", "", ""))
                    continue
                postal_key = (pin, row["officename"])
                if postal_key in overrides:
                    decision = overrides[postal_key]
                    if (row["statename"], row["district"]) != (decision["expected_state"], decision["expected_district"]):
                        raise ValueError(f"Postal source decision no longer matches {postal_key}")
                    for field, new_value in (("statename", decision["state"]), ("district", decision["district"])):
                        removed.append(("postal", number, "", field, row[field], new_value,
                                        "reference_backfill", "verified_postal_area", decision["source_url"],
                                        "", decision["note"]))
                        row[field] = new_value
                    applied.add(postal_key)
                    counts["reference_rows_corrected"] += 1
                group = groups[pin]
                group["rows"] += 1
                st = state(row["statename"])
                district = " ".join(clean(row["district"]).casefold().split())
                district = DISTRICT_ALIASES.get(district, district)
                if st:
                    group["states"].add(st)
                if st and district:
                    group["districts"].add((st, district))
                pt = point(row["latitude"], row["longitude"])
                if pt:
                    group["points"].add(pt)
                else:
                    counts["coordinate_pairs_unusable"] += 1
                    raw_pair = [clean(row[field]) for field in ("latitude", "longitude")]
                    if any(raw_pair):
                        removed.append(("postal", number, "", "latitude+longitude", json.dumps(raw_pair), "",
                                        "remove_field", "postal_coordinate_pair_unusable", "", "", ""))
                        counts["coordinate_pairs_removed"] += 1
                row = {field: clean(value) for field, value in row.items()}
                row["pincode"] = pin
                row["latitude"], row["longitude"] = pt if pt else ("", "")
                writer.writerow([*row.values(), number, st, district,
                                 "usable" if pt else "unusable"])
                counts["output_rows"] += 1

    def lookup_rows():
        for pin, group in sorted(groups.items()):
            state_n, district_n = len(group["states"]), len(group["districts"])
            counts["state_ambiguous_pins"] += int(state_n > 1)
            counts["district_ambiguous_pins"] += int(district_n > 1)
            yield (pin, group["rows"], json.dumps(sorted(group["states"])),
                   json.dumps(sorted(group["districts"]), ensure_ascii=False), len(group["points"]),
                   int(state_n > 1), int(district_n > 1))

    table(dest/"pincode_lookup.csv", fields, lookup_rows())
    counts["unique_pins"] = len(groups)
    if applied != set(overrides):
        raise ValueError(f"Unapplied postal overrides: {sorted(set(overrides) - applied)}")
    assert counts["input_rows"] == counts["output_rows"] + counts["invalid_pins_removed"] + counts["exact_duplicates_removed"]
    return groups, counts, removed


In [ ]:
staging_directory = TemporaryDirectory(prefix="datathon-cleaning-")
staging = Path(staging_directory.name)
postal_reference, postal_summary, postal_removals = index_post(raw_postal_csv, staging)
print(f"Cleaned {postal_summary['input_rows']:,} post office records to {postal_summary['output_rows']:,}.")
print(f"Built the reference for {len(postal_reference):,} unique PINs.")


## Silver · Clean facilities and service claims

- Check record identity and exclude invalid UUIDs, duplicate copies, and conflicting UUID groups.
- Normalize scalar fields, dates, URLs, and arrays. Deduplicate specialty capitalization variants.
- Apply reviewed source, website, and display-name decisions. Keep raw facility names for traceability.
- Check PIN and state consistency, screen coordinates, and flag points more than 50 km from the nearest same-PIN postal office.
- Record corrections, unresolved cases, supplied claims, and claims with supporting review evidence.


In [ ]:
def clean_facilities(header, records, prior_pins, prior_claims, reference, postal_counts,
                     postal_removals, input_fingerprints, dest):
    removed = list(postal_removals)
    with StringIO(FACILITY_SOURCE_DECISIONS_CSV, newline="") as f:
        decisions = defaultdict(list)
        seen_fields = set()
        for decision in csv.DictReader(f):
            key = (decision["id"], decision["field"])
            if key in seen_fields:
                raise ValueError(f"Duplicate hospital source decision: {key}")
            seen_fields.add(key)
            decisions[decision["id"]].append(decision)
    website_decisions = defaultdict(list)
    seen_websites = set()
    for decision in json.loads(WEBSITE_DECISIONS_JSON):
        key = (decision["id"], decision["field"], decision["original_value"])
        if key in seen_websites:
            raise ValueError(f"Duplicate website decision: {key}")
        seen_websites.add(key)
        website_decisions[decision["id"]].append(decision)
    name_decisions = read_name_decisions()
    by_id = defaultdict(list)
    counts, field_actions, pin_checks, claim_checks = Counter(), Counter(), Counter(), Counter()
    evidence_rows, services, queue = [], [], []
    for num, (row, start, end) in enumerate(records, 1):
        if not UUID_RE.fullmatch(row["unique_id"]):
            counts["invalid_uuid"] += 1
            removed.append(("hospital", num, "", "unique_id", "", "", "remove_row",
                            "invalid_uuid", "", "", f"Raw physical lines {start}–{end}"))
        else:
            by_id[row["unique_id"].lower()].append((num, row))
    cleaned_cols = [*header, "sanctioned_beds", "name_standardized", "source_record", "pin_status", "postal_state_candidates",
                    "postal_district_candidates", "nearest_post_office_km", "geography_status", "source_backfill_fields"]
    claim_cols = ["unique_id", "source_record", "field", "claim_text", "status"]
    known_states = set().union(*(g["states"] for g in reference.values()))
    applied_ids = set()
    count_output = 0
    with (dest/"facilities_clean.csv").open("w", encoding="utf-8-sig", newline="") as clean_file, \
         (dest/"facility_claims.csv").open("w", encoding="utf-8-sig", newline="") as claims_file:
        hospital_writer, claim_writer = csv.writer(clean_file), csv.writer(claims_file)
        hospital_writer.writerow(cleaned_cols)
        claim_writer.writerow(claim_cols)
        for uid, rows in by_id.items():
            number, original = rows[0]
            if any(r != original for _, r in rows):
                for idx, _ in rows:
                    removed.append(("hospital", idx, uid, "unique_id", uid, "", "remove_row",
                                    "conflicting_uuid", "", "", ""))
                    counts["conflicting_uuid_rows"] += 1
                continue
            for idx, _ in rows[1:]:
                counts["exact_duplicates_removed"] += 1
                removed.append(("hospital", idx, uid, "unique_id", uid, "", "remove_row",
                                "exact_duplicate", "", "", f"Retained logical record {number}"))
            row = {field: clean(val) for field, val in original.items()}
            row["sanctioned_beds"] = ""
            branch = (original["unique_id"], original["name"], original["address_city"], original["address_line1"])
            row["address_zipOrPostcode"] = re.sub(r"\s+", "", row["address_zipOrPostcode"])

            def change(field, value, reason, url="", second="", note=""):
                old = row[field]
                if old == value:
                    return
                field_actions[reason] += 1
                removed.append(("hospital", number, uid, field, original.get(field, old), value,
                                "source_backfill" if value else "remove_field", reason, url, second, note))
                row[field] = value

            repaired = []
            edits = decisions.get(uid, [])
            resulting_location = {field: row[field] for field in ("address_stateOrRegion", "address_zipOrPostcode")}
            edited_location = {edit["field"]: edit["value"] for edit in edits if edit["field"] in resulting_location}
            resulting_location.update(edited_location)
            if any(edited_location.values()):
                pin = resulting_location["address_zipOrPostcode"]
                st = state(resulting_location["address_stateOrRegion"])
                # Two sources can establish a branch PIN missing from this snapshot.
                # Leave its postal distance and consistency status unresolved.
                if pin not in reference:
                    if not (PIN.fullmatch(pin) and st in known_states and any(
                        edit["field"] == "address_zipOrPostcode"
                        and edit["status"] == "source_backfill_unlisted_pin"
                        and edit["source_url"] and edit["corroborating_url"]
                        and edit["source_url"] != edit["corroborating_url"]
                        for edit in edits
                    )):
                        raise ValueError(f"Unverified PIN outside postal snapshot for {uid}: {st} / {pin}")
                elif st not in reference[pin]["states"]:
                    raise ValueError(f"Source-backed state and PIN conflict for {uid}: {st} / {pin}")
            coordinate_conflict = any(d["status"] == "coordinate_location_conflict" for d in edits)
            for d in edits:
                field, old, new = d["field"], d["expected"], d["value"]
                if row[field] != old:
                    raise ValueError(f"Source decision key changed for {uid} / {field}")
                change(field, new, d["status"], d["source_url"], d["corroborating_url"], d["note"])
                applied_ids.add(uid)
                if new:
                    repaired.append(field)
                elif d["status"] != "coordinate_location_conflict":
                    queue.append((uid, field, d["status"], d["source_url"], d["note"]))
            if coordinate_conflict:
                field_actions["coordinate_locations_withdrawn"] += 1
                queue.append((uid, "coordinates", "coordinate_location_conflict", "",
                              "Original point was withdrawn. Obtain a branch-level location before distance ranking."))
            if row["address_zipOrPostcode"] and not PIN.fullmatch(row["address_zipOrPostcode"]):
                change("address_zipOrPostcode", "", "invalid_pin_format")
            if row["email"].casefold() in ("[email protected]", "[email\u00a0protected]"):
                change("email", "", "masked_email")
            for field in ("yearEstablished", "numberDoctors", "capacity", "area"):
                val = row[field]
                if val and not re.fullmatch(r"[0-9]+(?:\.[0-9]+)?", val):
                    change(field, "", "invalid_nonnegative_number")
                elif val and field == "yearEstablished" and (not float(val).is_integer() or not 1 <= float(val) <= int(AS_OF[:4])):
                    change(field, "", "invalid_establishment_year")
            for field in ("recency_of_page_update", "post_metrics_most_recent_social_media_post_date"):
                day = row[field][:10]
                try:
                    parsed_day = date.fromisoformat(day)
                except ValueError:
                    continue
                if parsed_day > date.fromisoformat(AS_OF):
                    change(field, "", "future_page_or_post_date")
            lat, lon = row["latitude"], row["longitude"]
            pt = point(lat, lon)
            if (lat or lon) and not pt:
                for field in ("latitude", "longitude", "coordinates"):
                    change(field, "", "coordinate_pair_unusable")

            verified_for_branch = []
            for field in ARRAY_FIELDS:
                raw_value = row[field]
                if not raw_value:
                    continue
                try:
                    values = json.loads(raw_value)
                except ValueError:
                    values = None
                if not isinstance(values, list):
                    change(field, "", "unparseable_list", note="Original value retained in the change log")
                    continue
                kept = []
                seen_text = set()
                for value in values:
                    if value is None or (isinstance(value, str) and not clean(value)):
                        field_actions["empty_array_items_removed"] += 1
                        continue
                    if not isinstance(value, str) and field in CLAIM_ARRAYS:
                        field_actions["nontext_claim_items_removed"] += 1
                        removed.append(("hospital", number, uid, field, json.dumps(value), "",
                                        "remove_claim", "nontext_claim_item", "", "", ""))
                        continue
                    normalized = clean(value) if isinstance(value, str) else value
                    key = json.dumps(normalized, ensure_ascii=False, sort_keys=True)
                    if key in seen_text:
                        field_actions["exact_array_duplicates_removed"] += 1
                        continue
                    seen_text.add(key)
                    proof = prior_claims.get((branch, field, normalized)) if field in CLAIM_ARRAYS else None
                    status = proof[0] if proof else "dataset_only"
                    if field in CLAIM_ARRAYS:
                        claim_checks[status] += 1
                        claim_writer.writerow([uid, number, field, normalized, status])
                    if proof:
                        evidence_rows.append((uid, field, normalized, *proof))
                    if status in EXCLUDED_CLAIMS:
                        field_actions["wrong_facility_claims_removed"] += 1
                        removed.append(("hospital", number, uid, field, normalized, "", "remove_claim",
                                        status, proof[1], "", proof[2]))
                        continue
                    kept.append(normalized)
                    if status in SUPPORTED_CLAIMS:
                        verified_for_branch.append((field, normalized, *proof))
                if field == "specialties":
                    unique_specialties = {}
                    duplicate_count = 0
                    for specialty in kept:
                        key = specialty.casefold()
                        if key in unique_specialties:
                            duplicate_count += 1
                            removed.append(("hospital", number, uid, field, specialty, unique_specialties[key],
                                            "remove_claim", "specialty_case_duplicate", "", "", ""))
                        else:
                            unique_specialties[key] = specialty
                    if duplicate_count:
                        field_actions["specialty_case_duplicates_removed"] += duplicate_count
                        field_actions["specialty_records_deduplicated"] += 1
                    kept = list(unique_specialties.values())
                row[field] = json.dumps(kept, ensure_ascii=False) if kept else ""
            for decision in website_decisions.pop(uid, ()):
                field = decision["field"]
                bad_url = decision["original_value"]
                retained_url = decision.get("retained_value", "")
                source_url = decision["source_url"]
                second_url = decision.get("corroborating_url", "")
                note = decision["note"]
                reason = "verified_website_duplicate_removed" if retained_url else "unverified_malformed_website_removed"
                if field == "officialWebsite" and not retained_url:
                    if row[field] != bad_url:
                        raise ValueError(f"Official website decision failed to match {uid}: {bad_url!r}")
                    change(field, "", reason, source_url, second_url, note)
                elif field == "websites":
                    urls = json.loads(row[field] or "[]")
                    if bad_url not in urls or (retained_url and (retained_url == bad_url or retained_url not in urls)):
                        raise ValueError(f"Website decision failed to match {uid}: {bad_url!r}, {retained_url!r}")
                    urls.remove(bad_url)
                    row[field] = json.dumps(urls, ensure_ascii=False) if urls else ""
                    field_actions[reason] += 1
                    removed.append(("hospital", number, uid, field, bad_url, retained_url,
                                    "remove_item", reason, source_url, second_url, note))
                else:
                    raise ValueError(f"Invalid website decision for {uid}: {field!r}")
            pin = row["address_zipOrPostcode"]
            p = reference.get(pin)
            st = state(row["address_stateOrRegion"])
            pin_status = "missing_or_removed" if not pin else "not_in_snapshot" if p is None else (
                "ambiguous_state" if len(p["states"]) > 1 else
                "state_unresolved" if not p["states"] or st not in known_states else
                "state_consistent" if st in p["states"] else "state_conflict")
            pin_checks[pin_status] += 1
            distance = min((km(pt, dst) for dst in p["points"]), default=None) if pt and p else None
            if coordinate_conflict:
                geo = "coordinate_location_conflict"
            elif distance is None:
                geo = "distance_unavailable"
            else:
                geo = "review_over_50_km" if distance > 50 else "postal_distance_within_50_km"
            if pin_status in {"not_in_snapshot", "ambiguous_state", "state_conflict"} or geo == "review_over_50_km":
                queue.append((uid, "geography", f"{pin_status} / {geo}", "", "Postal screening only"))
            if branch in prior_pins:
                prior, ref_pin, url, second, note = prior_pins[branch]
                if prior == "reference_sources_disagree" or (ref_pin and ref_pin != pin):
                    queue.append((uid, "address_zipOrPostcode", "prior_external_pin_needs_review", url, f"{ref_pin} {note}".strip()))
            if row["description"]:
                claim_checks["description_unreviewed"] += 1
                claim_writer.writerow([uid, number, "description", row["description"], "dataset_only"])
            for field, claim, status, url, note, reviewed_on in verified_for_branch:
                services.append((uid, row["name"], field, claim, status, url, note, reviewed_on, geo))
            display_name = " ".join(row["name"].split())
            name_decision = name_decisions.pop(uid, None)
            if name_decision is not None:
                if display_name != name_decision["expected_name"]:
                    raise ValueError(f"Facility name decision did not match: {uid}")
                cleaned_name = name_decision["cleaned_name"]
                field_actions["facility_display_names_cleaned"] += 1
                removed.append(("hospital", number, uid, "name_standardized", display_name, cleaned_name,
                                "normalize_display_name", "address_in_facility_name", name_decision["source_url"],
                                "", name_decision["note"]))
                display_name = cleaned_name
            hospital_writer.writerow([*(row[k] for k in [*header,"sanctioned_beds"]), display_name, number, pin_status,
                                       json.dumps(sorted(p["states"]) if p else [], ensure_ascii=False),
                                       json.dumps(sorted(p["districts"]) if p else [], ensure_ascii=False),
                                       round(distance, 1) if distance is not None else "", geo, "|".join(repaired)])
            count_output += 1
    assert applied_ids == set(decisions)
    if website_decisions:
        raise ValueError(f"Website decisions did not match facilities: {list(website_decisions)}")
    if name_decisions:
        raise ValueError(f"Name decisions did not match facilities: {list(name_decisions)}")
    assert len(records) == count_output + sum(counts.values())
    assert sum(pin_checks.values()) == count_output
    log_cols = ("table", "source_record", "unique_id", "field", "original_value", "cleaned_value",
                "action", "reason", "source_url", "corroborating_url", "note")
    table(dest/"removal_log.csv", log_cols, removed)
    table(dest/"backfills.csv", log_cols, [r for r in removed if r[6] == "source_backfill"])
    table(dest/"postal_backfills.csv", log_cols, [r for r in removed if r[6] == "reference_backfill"])
    table(dest/"review_queue.csv", ("unique_id", "field", "reason", "source_url", "note"), queue)
    table(dest/"reviewed_claim_evidence.csv", ("unique_id","field","claim","status","url","note","reviewed_on"), evidence_rows)
    table(dest/"verified_service_claims.csv", ("unique_id","name","field","claim","status","source_url","note","reviewed_on","geography_status"),services)
    summary = {
        "as_of_date":AS_OF,"hospital_input_rows":len(records),"hospital_output_rows":count_output,
        "hospital_removed_rows":dict(counts),"post":dict(postal_counts),"field_actions":dict(field_actions),
        "pin_checks_after_cleaning":dict(pin_checks),"claims":dict(claim_checks),
        "verified_service_claim_rows":len(services),"review_queue_rows":len(queue),
        "dataset_sha256": input_fingerprints,
    }
    (dest/"summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    return summary


In [ ]:
summary = clean_facilities(
    facility_columns, bronze_facilities, prior_pins, prior_claims, postal_reference,
    postal_summary, postal_removals, input_fingerprints, staging,
)
print(json.dumps(summary, ensure_ascii=False, indent=2))


## Gold · Reconcile counts and compare data quality

The comparison uses the same cleaned postal reference and the same checks for both snapshots. Coordinates must be finite and within the pipeline's screening bounds. Valid PINs must have six digits and appear in the reference. Region match requires one postal state that agrees with the facility state. Completeness requires name, street, city, state, PIN, latitude, and longitude.


In [ ]:
CORE_FIELDS = ("name", "address_line1", "address_city", "address_stateOrRegion",
               "address_zipOrPostcode", "latitude", "longitude")

def quality_counts(rows):
    counts = [0, 0, 0, 0]
    total = 0
    for row in rows:
        pin = re.sub(r"\s+", "", clean(row["address_zipOrPostcode"]))
        valid_pin = PIN.fullmatch(pin) is not None and pin in postal_reference
        counts[0] += point(clean(row["latitude"]), clean(row["longitude"])) is not None
        counts[1] += valid_pin
        counts[2] += valid_pin and postal_reference[pin]["states"] == {state(row["address_stateOrRegion"])}
        counts[3] += all(clean(row[field]) for field in CORE_FIELDS)
        total += 1
    return total, counts

before_total, before_counts = quality_counts(row for row, _, _ in bronze_facilities)
with (staging / "facilities_clean.csv").open(encoding="utf-8-sig", newline="") as file:
    after_total, after_counts = quality_counts(csv.DictReader(file))

if before_total != summary["hospital_input_rows"] or after_total != summary["hospital_output_rows"]:
    raise ValueError("Facility row counts do not reconcile with the run summary")
if after_counts[2] != summary["pin_checks_after_cleaning"].get("state_consistent", 0):
    raise ValueError("The state match count does not reconcile with the run summary")

metric_labels = ("Valid coordinates", "Valid PIN", "PIN and state agree", "Key fields complete")
quality_report = []
for metric, before_count, after_count in zip(metric_labels, before_counts, after_counts):
    before_pct = 100 * before_count / before_total
    after_pct = 100 * after_count / after_total
    quality_report.append((metric, before_count, before_total, after_count, after_total,
                           round(before_pct, 2), round(after_pct, 2), round(after_pct - before_pct, 2)))

metric_columns = ["metric", "before_count", "before_total", "after_count", "after_total",
                  "before_pct", "after_pct", "change_pp"]
if "spark" in globals() and "display" in globals():
    display(spark.createDataFrame(quality_report, metric_columns))
else:
    for row in quality_report:
        print(dict(zip(metric_columns, row)))


## Gold · Save and review the outputs

The run directory contains the cleaned facilities, postal offices and PIN lookup, the full claim audit, selected claim evidence, source-backed changes, the removal log, and the review queue. The summary records the original input fingerprints and counts by status.


In [ ]:
run_id = datetime.now(timezone.utc).strftime("run-%Y%m%dT%H%M%S%fZ")
run_path = output_root / run_id
run_path.mkdir(parents=True, exist_ok=False)

# Write the summary last so it identifies a complete export.
for product in sorted(staging.iterdir()):
    if product.name != "summary.json":
        copyfile(product, run_path / product.name)
copyfile(staging / "summary.json", run_path / "summary.json")

print(f"Saved {len(list(run_path.iterdir()))} files to {run_path}")
for path in sorted(run_path.iterdir()):
    print(path.name)
